# Archived alternative MosMed transfer\n\nAlternative MosMed transfer configuration retained for thesis provenance. This public notebook retains the thesis research implementation, but the clinical dataset, derived volumes, annotations, labels, identifiers, checkpoints, and executed outputs are not included.\n\n## Configuration\nSet the paths below only to data for which you have appropriate authorisation. Do not commit local paths or generated clinical outputs.\n

In [ ]:
from pathlib import Path\n\nDATA_ROOT = Path(\"/path/to/authorised/data\")\nOUTPUT_ROOT = Path(\"./outputs\")\nOUTPUT_ROOT.mkdir(parents=True, exist_ok=True)\n

In [ ]:
# Google Colab Drive import removed for the public copy.\n# Configure DATA_ROOT below instead of mounting a private drive.


In [ ]:
# 06 - Build one unified training index (manifest + labels + folds) + outlier removal + log target

import pandas as pd
import numpy as np

OUT_ROOT = str(DATA_ROOT)
MANIFEST_PATH = f"{OUT_ROOT}/volume_manifest_.csv"
SPLITS_PATH   = f"{OUT_ROOT}/splits_5fold_locked.csv"
MASTER_PATH   = f"{OUT_ROOT}/master_patient_index.csv"

man = pd.read_csv(MANIFEST_PATH)
man["patient_id"] = man["patient_id"].astype(str).str.zfill(4)
man = man.query("status == 'OK'").copy()

spl = pd.read_csv(SPLITS_PATH)
spl["patient_id"] = spl["patient_id"].astype(str).str.zfill(4)
spl["fold"] = spl["fold"].astype(int)

master = pd.read_csv(MASTER_PATH)
master["patient_id"] = master["patient_id"].astype(str).str.zfill(4)
master["label"] = master["label"].astype(float)

# Uniqueness checks
assert man["patient_id"].duplicated().sum() == 0, "Manifest has duplicate OK patient rows."
assert spl["patient_id"].duplicated().sum() == 0, "Splits file has duplicate patient rows."
assert master["patient_id"].duplicated().sum() == 0, "Master has duplicate patient rows."

# keep only patients that exist in splits_5fold_locked.csv
spl_ids = set(spl["patient_id"].tolist())
man = man[man["patient_id"].isin(spl_ids)].copy()
master = master[master["patient_id"].isin(spl_ids)].copy()

# Merge
df_index = (man
    .merge(master[["patient_id","label"]], on="patient_id", how="left", validate="one_to_one")
    .merge(spl[["patient_id","fold"]],     on="patient_id", how="left", validate="one_to_one")
)


assert df_index["label"].notna().all(), "Some labels missing after merge."
assert df_index["fold"].notna().all(),  "Some folds missing after merge (check splits file)."
assert df_index["volume_path"].astype(str).str.len().gt(0).all(), "Missing volume_path."
assert df_index["patient_id"].nunique() == len(df_index), "train_index has duplicate patients."

# Outlier removal
OUTLIER_PIDS = set([\"REDACTED_PATIENT_ID\", \"REDACTED_PATIENT_ID\", \"REDACTED_PATIENT_ID\"])
df_index["is_outlier"] = df_index["patient_id"].isin(OUTLIER_PIDS)
df_index = df_index[~df_index["is_outlier"]].copy()

# LOG TARGET: log10(label); if label==0 -> keep 0)
df_index["label_raw"] = df_index["label"].astype(float)
df_index["label_log10"] = np.where(
    df_index["label_raw"] > 0,
    np.log10(df_index["label_raw"]),
    0.0
).astype(float)

df_index = df_index.sort_values("patient_id").reset_index(drop=True)

print("Training index rows:", len(df_index))
display(df_index[["patient_id","label_raw","label_log10","fold","n_slices"]].head(12))

INDEX_PATH = f"{OUT_ROOT}/train_index.csv"
df_index.to_csv(INDEX_PATH, index=False)
print("Saved:", INDEX_PATH)

# Folds distribution
print("\nFold counts:")
print(df_index["fold"].value_counts().sort_index())


In [ ]:
# Cell A — MosMed train/val/test split (patient-level, reproducible)

import os
import numpy as np
import pandas as pd
from pathlib import Path

PROJECT_ROOT = DATA_ROOT
MOSMED_INDEX = os.path.join(PROJECT_ROOT, "external_data/mosmed/mosmed_index.csv")
OUT_SPLIT = os.path.join(PROJECT_ROOT, "external_data/mosmed/mosmed_split.csv")

SEED = 42
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15
assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-6

df = pd.read_csv(MOSMED_INDEX).copy()
df["patient_id"] = df["patient_id"].astype(str)

# patient-level shuffle
pids = np.array(sorted(df["patient_id"].unique()))
rng = np.random.default_rng(SEED)
rng.shuffle(pids)

n = len(pids)
n_train = int(round(n * TRAIN_FRAC))
n_val   = int(round(n * VAL_FRAC))
n_test  = n - n_train - n_val

train_p = set(pids[:n_train])
val_p   = set(pids[n_train:n_train+n_val])
test_p  = set(pids[n_train+n_val:])

def _assign(pid):
    if pid in train_p: return "train"
    if pid in val_p:   return "val"
    return "test"

df["split"] = df["patient_id"].map(_assign)

Path(os.path.dirname(OUT_SPLIT)).mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_SPLIT, index=False)

print("Saved:", OUT_SPLIT)
print("Patients total:", n, "| train:", len(train_p), "| val:", len(val_p), "| test:", len(test_p))
print("Class counts (patients) by split:")
print(df.drop_duplicates("patient_id").groupby(["split","severity_cat"]).size().unstack(fill_value=0))


In [ ]:
# Cell 07 — Preprocessing utilities (2-window + optional HF/spot channel, normalize, resize)
# Supports: highpass, DoG, LoG, DCT band-pass + lung-masked HF.
# small robustness improvements for external validation (MosMed etc.)

!pip -q install opencv-python-headless

from __future__ import annotations

import numpy as np
import cv2

IMG_SIZE = 256

# HU windows
LUNG_MIN, LUNG_MAX = -1350, 150
MEDIA_MIN, MEDIA_MAX = -160, 240

# Canonical HU clamp to reduce domain differences (scanner-dependent floors/ceilings)
HU_CANON_MIN, HU_CANON_MAX = -1350, 600


# -----------------------------
# HF/spot channel configuration
# -----------------------------
ADD_HF_CHANNEL: bool = True

# HF modes:
#   - "highpass":      I - Gσ(I)                         (simple high-frequency emphasis)
#   - "dog":           Gσ1(I) - Gσ2(I), σ2>σ1            (band-pass, scale-selective texture emphasis)
#   - "log":           LoG approx (Gaussian + Laplacian)  (blob/edge emphasis; can be vessel-heavy on CT)
#   - "dct_bandpass":  DCT -> zero low-freq -> iDCT      (real-valued frequency-space "remove low")
HF_MODE: str = "dct_bandpass"          # {"highpass", "dog", "log", "dct_bandpass"}
HF_FROM: str = "medi"              # {"medi", "lung"}

# Highpass params
HF_GAUSS_SIGMA: float = 1.6

# DoG params (must satisfy DOG_SIGMA2 > DOG_SIGMA1 > 0)
DOG_SIGMA1: float = 0.5
DOG_SIGMA2: float = 2.0

# LoG params
HF_LOG_SIGMA: float = 2.0

# DCT band-pass params
DCT_LOW_CUTOFF: int = 16                 # remove low-freq k×k block
DCT_HIGH_CUTOFF: int | None = None       # keep only top-left K×K band (optional)
DCT_USE_ABS: bool = True                 # abs(iDCT) -> "detail energy"

# Robust normalization percentiles
HF_PCTL_LOW: float = 1.0
HF_PCTL_HIGH: float = 99.0

# Safety: avoid NaNs/Infs ever reaching the model
SAFE_NAN_TO_NUM: bool = True


# ----------------------------------------
# Lung-mask configuration (HF-only masking)
# ----------------------------------------
USE_LUNG_MASK_FOR_HF: bool = True

LUNG_MASK_HU_THR: float = -400.0
LUNG_MASK_MIN_AREA: int = 1500
LUNG_MASK_CLOSE_KERNEL: int = 9
LUNG_MASK_DILATE_ITERS: int = 1
LUNG_MASK_EDGE_BLUR_SIGMA: float = 1.2


def hu_canonicalize(x_hu: np.ndarray) -> np.ndarray:
    """Clamp to a plausible HU range so scans share the same floor/ceiling."""
    x = np.clip(x_hu, HU_CANON_MIN, HU_CANON_MAX).astype(np.float32)
    if SAFE_NAN_TO_NUM:
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return x


def _window_norm(x_hu: np.ndarray, hu_min: float, hu_max: float) -> np.ndarray:
    """Clip to a HU window and normalize to [0, 1]."""
    x = np.clip(x_hu, hu_min, hu_max).astype(np.float32)
    x = (x - hu_min) / (hu_max - hu_min + 1e-6)
    if SAFE_NAN_TO_NUM:
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return x


def _resize01(x: np.ndarray, size: int = IMG_SIZE) -> np.ndarray:
    """Resize to (size, size) and clamp to [0, 1]."""
    x = cv2.resize(x, (size, size), interpolation=cv2.INTER_AREA)
    x = np.clip(x, 0.0, 1.0).astype(np.float32)
    if SAFE_NAN_TO_NUM:
        x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return x


def _robust_rescale_01(x: np.ndarray, p_low: float = HF_PCTL_LOW, p_high: float = HF_PCTL_HIGH) -> np.ndarray:
    """
    Robustly rescale an array to [0, 1] using percentiles.
    Helps prevent outliers (metal/streak artifacts) from dominating.
    """
    x = np.asarray(x, dtype=np.float32)
    lo, hi = np.percentile(x, [p_low, p_high])
    y = np.clip((x - lo) / (hi - lo + 1e-6), 0.0, 1.0).astype(np.float32)
    if SAFE_NAN_TO_NUM:
        y = np.nan_to_num(y, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    return y


def _hf_channel(img01: np.ndarray, mode: str = HF_MODE) -> np.ndarray:
    """
    Build a high-frequency / 'spot emphasis' channel from a [0,1] image.
    Output: [0,1] float32, robustly normalized.
    """
    img = img01.astype(np.float32)

    if mode == "highpass":
        blur = cv2.GaussianBlur(img, (0, 0), sigmaX=HF_GAUSS_SIGMA, sigmaY=HF_GAUSS_SIGMA)
        return _robust_rescale_01(img - blur)

    if mode == "dog":
        s1 = float(DOG_SIGMA1)
        s2 = float(DOG_SIGMA2)
        if not (s2 > s1 > 0):
            raise ValueError(f"DoG requires DOG_SIGMA2 > DOG_SIGMA1 > 0. Got DOG_SIGMA1={s1}, DOG_SIGMA2={s2}.")
        g1 = cv2.GaussianBlur(img, (0, 0), sigmaX=s1, sigmaY=s1)
        g2 = cv2.GaussianBlur(img, (0, 0), sigmaX=s2, sigmaY=s2)
        return _robust_rescale_01(g1 - g2)

    if mode == "log":
        sm = cv2.GaussianBlur(img, (0, 0), sigmaX=HF_LOG_SIGMA, sigmaY=HF_LOG_SIGMA)
        log = cv2.Laplacian(sm, ddepth=cv2.CV_32F, ksize=3)
        return _robust_rescale_01(log)

    if mode == "dct_bandpass":
        k = int(DCT_LOW_CUTOFF)
        if k < 0:
            raise ValueError(f"DCT_LOW_CUTOFF must be >= 0. Got {k}.")

        # Small robustness: remove global offset before DCT (helps cross-domain stability)
        mu = float(np.mean(img))
        img0 = (img - mu).astype(np.float32)

        coeff = cv2.dct(img0)

        if k > 0:
            kk = min(k, coeff.shape[0], coeff.shape[1])
            coeff[:kk, :kk] = 0.0

        if DCT_HIGH_CUTOFF is not None:
            K = int(DCT_HIGH_CUTOFF)
            if K <= 0:
                raise ValueError(f"DCT_HIGH_CUTOFF must be > 0 when provided. Got {K}.")
            K = min(K, coeff.shape[0], coeff.shape[1])
            coeff2 = np.zeros_like(coeff)
            coeff2[:K, :K] = coeff[:K, :K]
            coeff = coeff2

        recon = cv2.idct(coeff).astype(np.float32) + mu  # restore mean

        if bool(DCT_USE_ABS):
            recon = np.abs(recon)

        return _robust_rescale_01(recon)

    raise ValueError(f"Unknown HF_MODE={mode!r}. Choose from: 'highpass', 'dog', 'log', 'dct_bandpass'.")


def _lung_mask_from_hu(slice_hu: np.ndarray) -> np.ndarray:
    """
    Build a crude lung mask from a single HU slice.
    Note: This is a biasing mask (not medical-grade segmentation).
    """
    x = slice_hu.astype(np.float32)
    air = (x < float(LUNG_MASK_HU_THR)).astype(np.uint8)

    h, w = air.shape
    flood = air.copy()
    ff_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)

    seeds = [
        (0, 0), (0, w - 1), (h - 1, 0), (h - 1, w - 1),
        (0, w // 2), (h - 1, w // 2), (h // 2, 0), (h // 2, w - 1)
    ]
    for sy, sx in seeds:
        if flood[sy, sx] == 1:
            cv2.floodFill(flood, ff_mask, seedPoint=(sx, sy), newVal=2)

    background = (flood == 2)
    internal_air = air.copy()
    internal_air[background] = 0

    num, labels, stats, _ = cv2.connectedComponentsWithStats(internal_air, connectivity=8)
    keep = np.zeros_like(internal_air, dtype=np.uint8)
    for lab in range(1, num):
        area = int(stats[lab, cv2.CC_STAT_AREA])
        if area >= int(LUNG_MASK_MIN_AREA):
            keep[labels == lab] = 1

    if int(LUNG_MASK_CLOSE_KERNEL) >= 3:
        k = int(LUNG_MASK_CLOSE_KERNEL)
        if k % 2 == 0:
            k += 1
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
        keep = cv2.morphologyEx(keep, cv2.MORPH_CLOSE, kernel)

    if int(LUNG_MASK_DILATE_ITERS) > 0:
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        keep = cv2.dilate(keep, kernel, iterations=int(LUNG_MASK_DILATE_ITERS))

    return keep.astype(np.float32)


def preprocess_slice(slice_hu: np.ndarray) -> np.ndarray:
    """
    Convert a 2D HU slice into a model-ready tensor.

    Returns:
        x: (C, H, W) float32
           - C=2 -> [lung, medi]
           - C=3 -> [lung, medi, hf] if ADD_HF_CHANNEL=True
    """
    slice_hu = hu_canonicalize(slice_hu)

    lung = _resize01(_window_norm(slice_hu, LUNG_MIN, LUNG_MAX))
    medi = _resize01(_window_norm(slice_hu, MEDIA_MIN, MEDIA_MAX))

    chans = [lung, medi]

    if ADD_HF_CHANNEL:
        base = medi if HF_FROM == "medi" else lung
        hf = _hf_channel(base, mode=HF_MODE)

        if USE_LUNG_MASK_FOR_HF:
            m = _lung_mask_from_hu(slice_hu)  # (H,W) in {0,1}
            m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)
            m = np.clip(m, 0.0, 1.0)

            if float(LUNG_MASK_EDGE_BLUR_SIGMA) > 0:
                m = cv2.GaussianBlur(m, (0, 0),
                                     sigmaX=float(LUNG_MASK_EDGE_BLUR_SIGMA),
                                     sigmaY=float(LUNG_MASK_EDGE_BLUR_SIGMA))
                m = np.clip(m, 0.0, 1.0)

            hf = hf * m

        chans.append(hf)

    return np.stack(chans, axis=0).astype(np.float32)



In [ ]:
# Cell 08 — Representative slice sampling (QC filter + diversity + informative extremes)
# Updated: optional debug info + more deterministic internal seeding + small-kept guard

import numpy as np

CENTER_FRACTION = 0.75
N_SLICES_REP    = 160
FEAT_HW         = 64
N_PCA           = 24
RANDOM_STATE    = 42

LUNG_HU_THR    = -350
MIN_LUNG_FRAC  = 0.06
MAX_AIR_FRAC   = 0.98
HI_DENS_THR    = -200
N_INFORMATIVE  = 30
MAX_BODY_FRAC  = 0.45

HU_CANON_MIN, HU_CANON_MAX = -1350, 600


def _central_candidates(n_slices: int, center_fraction: float):
    if n_slices <= 0:
        return np.array([], dtype=int)
    if center_fraction < 1.0:
        margin = int((1 - center_fraction) * n_slices / 2)
        lo = max(margin, 0)
        hi = max(n_slices - margin, lo + 1)
    else:
        lo, hi = 0, n_slices
    return np.arange(lo, hi, dtype=int)


def _resize_nearest(x2d: np.ndarray, out_hw: int):
    """Simple stride resize; fast and stable enough for diversity features."""
    H, W = x2d.shape
    sh = max(1, H // out_hw)
    sw = max(1, W // out_hw)
    y = x2d[::sh, ::sw][:out_hw, :out_hw]
    if y.shape != (out_hw, out_hw):
        out = np.zeros((out_hw, out_hw), dtype=y.dtype)
        out[:y.shape[0], :y.shape[1]] = y
        y = out
    return y


def _ensure_k_unique(idx: np.ndarray, candidates: np.ndarray, k: int, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    idx = np.unique(idx.astype(int))
    candidates = np.unique(candidates.astype(int))
    idx = idx[np.isin(idx, candidates)]

    if len(candidates) <= k:
        return np.sort(candidates)

    if len(idx) >= k:
        keep = np.linspace(0, len(idx) - 1, k).round().astype(int)
        return np.sort(idx[keep])

    need = k - len(idx)
    remaining = candidates[~np.isin(candidates, idx)]
    if len(remaining) > 0:
        take = min(need, len(remaining))
        fill = rng.choice(remaining, size=take, replace=False)
        idx = np.unique(np.concatenate([idx, fill]))

    if len(idx) < k:
        need = k - len(idx)
        base = np.linspace(0, len(candidates) - 1, need).round().astype(int)
        idx = np.unique(np.concatenate([idx, candidates[base]]))

    if len(idx) > k:
        keep = np.linspace(0, len(idx) - 1, k).round().astype(int)
        idx = idx[keep]

    return np.sort(idx.astype(int))


def _slice_qc_metrics(slice_hu: np.ndarray):
    """
    Cheap QC + informativeness metrics from HU slice.
    Returns:
      lung_frac, air_frac, hi_dens_frac_in_lung, body_frac, mean_hu, std_hu
    """
    x = np.clip(slice_hu, HU_CANON_MIN, HU_CANON_MAX).astype(np.float32)

    lung_mask = (x < LUNG_HU_THR)
    lung_frac = float(lung_mask.mean())
    air_frac = float((x < -950).mean())

    if lung_mask.any():
        hi_dens_frac = float((x[lung_mask] > HI_DENS_THR).mean())
    else:
        hi_dens_frac = 0.0

    body_frac = float((x > -300).mean())

    mean_hu = float(np.mean(x))
    std_hu  = float(np.std(x))
    return lung_frac, air_frac, hi_dens_frac, body_frac, mean_hu, std_hu


def sample_slice_indices_representative(
    vol: np.ndarray,
    n_select: int = N_SLICES_REP,
    center_fraction: float = CENTER_FRACTION,
    rng=None,
    feat_hw: int = FEAT_HW,
    n_pca: int = N_PCA,
    n_informative: int = N_INFORMATIVE,
    return_debug: bool = False,
):
    """
    Representative sampling:
      1) central candidates
      2) QC filter (lung_frac/air_frac/body_frac)
      3) always include top informative slices (hi_dens_frac in lung)
      4) fill remaining with diversity clustering (PCA + MiniBatchKMeans)

    If return_debug=True, returns (idx, debug_dict).
    """
    rng = np.random.default_rng(RANDOM_STATE) if rng is None else rng
    seed_local = int(rng.integers(1_000_000_000))  # one seed for internal steps

    Z = int(vol.shape[0])
    candidates = _central_candidates(Z, center_fraction)

    if len(candidates) == 0:
        out = np.array([max(0, Z // 2)], dtype=int)
        return (out, {"reason": "no_candidates", "Z": Z}) if return_debug else out

    if len(candidates) <= n_select:
        out = candidates.astype(int)
        return (out, {"reason": "candidates_leq_n_select", "Z": Z, "n": len(out)}) if return_debug else out

    metrics = []
    kept = []
    for z in candidates:
        s = vol[int(z)]
        lung_frac, air_frac, hi_dens_frac, body_frac, mean_hu, std_hu = _slice_qc_metrics(s)

        if lung_frac < MIN_LUNG_FRAC:
            continue
        if air_frac > MAX_AIR_FRAC:
            continue
        if body_frac > MAX_BODY_FRAC:
            continue

        kept.append(int(z))
        metrics.append((lung_frac, air_frac, hi_dens_frac, body_frac, mean_hu, std_hu))

    if len(kept) == 0:
        base = np.linspace(0, len(candidates) - 1, n_select).round().astype(int)
        out = np.sort(candidates[base]).astype(int)
        return (out, {"reason": "qc_filtered_all", "Z": Z, "candidates": len(candidates)}) if return_debug else out

    kept = np.asarray(kept, dtype=int)
    metrics = np.asarray(metrics, dtype=np.float32)

    if len(kept) <= n_select:
        out = np.sort(kept).astype(int)
        return (out, {"reason": "qc_kept_leq_n_select", "Z": Z, "kept": len(kept)}) if return_debug else out

    # informative extremes
    hi_dens = metrics[:, 2]
    n_inf = int(min(n_informative, len(kept), max(8, n_select // 6)))
    informative_idx = np.argsort(-hi_dens)[:n_inf]
    chosen = kept[informative_idx]

    # if not enough kept for clustering to be meaningful, fill evenly
    remaining_k = int(max(0, n_select - len(np.unique(chosen))))
    if remaining_k <= 0:
        out = _ensure_k_unique(chosen, kept, n_select, rng=rng)
        return (out, {"reason": "informative_only", "Z": Z, "kept": len(kept), "n_inf": n_inf}) if return_debug else out

    # diversity clustering
    xs = vol[kept].astype(np.float32)
    xs = np.clip(xs, HU_CANON_MIN, 400.0)
    xs = (xs - HU_CANON_MIN) / (400.0 - HU_CANON_MIN + 1e-6)

    feats = np.stack([_resize_nearest(xi, feat_hw).reshape(-1) for xi in xs], axis=0)
    feats = feats - feats.mean(axis=0, keepdims=True)
    feats = feats / (feats.std(axis=0, keepdims=True) + 1e-6)

    # If too few samples, skip PCA/KMeans and fill evenly
    if feats.shape[0] < max(10, remaining_k + 2):
        base = np.linspace(0, len(kept) - 1, remaining_k).round().astype(int)
        idx = np.unique(np.concatenate([chosen, kept[base]]))
        out = _ensure_k_unique(idx, candidates=kept, k=n_select, rng=rng)
        return (out, {"reason": "too_few_for_kmeans", "Z": Z, "kept": len(kept), "n_inf": n_inf}) if return_debug else out

    feats_small = feats
    try:
        from sklearn.decomposition import PCA
        pca = PCA(
            n_components=min(n_pca, feats.shape[1], feats.shape[0] - 1),
            random_state=seed_local,
        )
        feats_small = pca.fit_transform(feats)
    except Exception:
        feats_small = feats

    try:
        from sklearn.cluster import MiniBatchKMeans
        from sklearn.metrics import pairwise_distances_argmin_min

        km = MiniBatchKMeans(
            n_clusters=remaining_k,
            random_state=seed_local,
            batch_size=1024,
            n_init="auto",
            max_iter=200,
        )
        km.fit(feats_small)
        nearest, _ = pairwise_distances_argmin_min(km.cluster_centers_, feats_small)
        diverse = kept[nearest]

        idx = np.unique(np.concatenate([chosen, diverse]))
        out = _ensure_k_unique(idx, candidates=kept, k=n_select, rng=rng)

        dbg = {"reason": "kmeans", "Z": Z, "candidates": len(candidates), "kept": len(kept), "n_inf": n_inf, "remaining_k": remaining_k}
        return (out, dbg) if return_debug else out

    except Exception as e:
        print("[warn] rep sampling clustering failed:", str(e))
        need = remaining_k
        base = np.linspace(0, len(kept) - 1, need).round().astype(int)
        idx = np.unique(np.concatenate([chosen, kept[base]]))
        out = _ensure_k_unique(idx, candidates=kept, k=n_select, rng=rng)
        return (out, {"reason": "kmeans_failed", "Z": Z, "kept": len(kept), "n_inf": n_inf}) if return_debug else out



In [ ]:
# Cell 08B — Visualization of representative selection per patient (lung + medi + filter + optional mask diagnostics)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2

IDX_PATH = f"{OUT_ROOT}/train_index.csv"
dfi = pd.read_csv(IDX_PATH)
dfi["patient_id"] = dfi["patient_id"].astype(str).str.zfill(4)


def _hf_label() -> str:
    """
    Human-readable HF label for plots, including mode-specific parameters.
    This makes screenshots/figures reproducible for thesis write-up.
    """
    mode = str(globals().get("HF_MODE", "NA"))
    src = str(globals().get("HF_FROM", "NA"))

    if mode == "highpass":
        sig = globals().get("HF_GAUSS_SIGMA", "NA")
        return f"HF(highpass σ={sig}, from={src})"

    if mode == "dog":
        s1 = globals().get("DOG_SIGMA1", "NA")
        s2 = globals().get("DOG_SIGMA2", "NA")
        return f"HF(dog σ1={s1}, σ2={s2}, from={src})"

    if mode == "log":
        sig = globals().get("HF_LOG_SIGMA", "NA")
        return f"HF(log σ={sig}, from={src})"

    if mode == "dct_bandpass":
        low = globals().get("DCT_LOW_CUTOFF", "NA")
        high = globals().get("DCT_HIGH_CUTOFF", None)
        use_abs = globals().get("DCT_USE_ABS", False)
        return f"HF(dct low={low}, high={high}, abs={use_abs}, from={src})"

    return f"HF({mode}, from={src})"


def plot_rep_selection_for_patient(pid, n_show=18, seed=42, show_mask_debug=True):
    """
    Visualize representative z coverage + the actual model inputs for selected slices.

    Rows show the channels produced by preprocess_slice:
      - lung window (normalized)
      - mediastinum window (normalized)
      - HF channel (if ADD_HF_CHANNEL=True)

    If show_mask_debug=True and USE_LUNG_MASK_FOR_HF=True:
      adds extra rows for:
        - lung mask (soft, resized)
        - HF before masking
        - HF after masking (should match preprocess HF)
    """
    pid = str(pid).zfill(4)
    row = dfi[dfi["patient_id"] == pid].iloc[0]
    vp = str(row["volume_path"])
    vol = np.load(vp, mmap_mode="r")
    Z = int(vol.shape[0])

    # Representative selection
    rng = np.random.default_rng(seed)
    idx = sample_slice_indices_representative(
        vol, n_select=N_SLICES_REP, center_fraction=CENTER_FRACTION, rng=rng
    )
    idx = np.array(sorted(set(map(int, idx))), dtype=int)

    # Print RAW label
    label_raw = row.get("label_raw", row.get("label", "NA"))
    label_log = row.get("label_log10", "NA")
    print(f"Patient {pid} | label_raw={label_raw} | label_log10={label_log} | Z={Z} | selected={len(idx)}")
    print("z_min/z_max:", int(idx.min()), int(idx.max()))
    print("HF config:", _hf_label(), "| lung_mask_for_hf=", bool(globals().get("USE_LUNG_MASK_FOR_HF", False)))

    # Coverage plot
    plt.figure(figsize=(10, 1.6))
    plt.scatter(idx, np.zeros_like(idx), s=15)
    plt.yticks([])
    plt.xlabel("z index")
    plt.title(f"{pid} representative z coverage")
    plt.show()

    # Choose evenly spaced subset from selected
    if len(idx) > n_show:
        keep = np.linspace(0, len(idx) - 1, n_show).round().astype(int)
        zs = idx[keep]
    else:
        zs = idx

    # Determine channels from preprocess_slice on first shown slice
    x0 = preprocess_slice(vol[int(zs[0])])
    x0 = np.asarray(x0)
    if x0.ndim == 2:
        x0 = x0[None, :, :]
    C = int(x0.shape[0])

    # Base channel names
    ch_names = ["lung", "mediastinum"]
    if C >= 3:
        ch_names.append(_hf_label())
    for k in range(3, C):
        ch_names.append(f"ch{k+1}")

    # mask debug rows
    can_mask_debug = (
        bool(show_mask_debug)
        and ("_lung_mask_from_hu" in globals())
        and bool(globals().get("USE_LUNG_MASK_FOR_HF", False))
        and (C >= 3)
    )

    extra_names = []
    extra_rows_per_slice = 0
    if can_mask_debug:
        extra_names = ["lung_mask (soft)", "HF (pre-mask)", "HF (masked)"]
        extra_rows_per_slice = 3

    total_rows = C + extra_rows_per_slice
    n = len(zs)

    fig, axes = plt.subplots(total_rows, n, figsize=(2.2 * n, 2.2 * total_rows))
    if total_rows == 1:
        axes = np.array([axes])
    if n == 1:
        axes = axes.reshape(total_rows, 1)

    for c_i, z in enumerate(zs):
        sl_hu = vol[int(z)]

        # model input channels
        x = preprocess_slice(sl_hu)
        x = np.asarray(x)
        if x.ndim == 2:
            x = x[None, :, :]

        # plot base channels
        for r_i in range(C):
            ax = axes[r_i, c_i]
            ax.imshow(x[r_i], cmap="gray", vmin=0.0, vmax=1.0)
            ax.set_title(f"{ch_names[r_i]}\nz={int(z)}", fontsize=10)
            ax.axis("off")

        # plot mask debug rows if enabled
        if can_mask_debug:
            # build soft resized mask exactly like preprocess does
            m = _lung_mask_from_hu(hu_canonicalize(sl_hu))  # (H,W) {0,1}
            m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)
            m = np.clip(m, 0.0, 1.0)
            if float(LUNG_MASK_EDGE_BLUR_SIGMA) > 0:
                m = cv2.GaussianBlur(
                    m, (0, 0),
                    sigmaX=float(LUNG_MASK_EDGE_BLUR_SIGMA),
                    sigmaY=float(LUNG_MASK_EDGE_BLUR_SIGMA)
                )
                m = np.clip(m, 0.0, 1.0)

            # reconstruct HF pre-mask and masked (verify masking behavior)
            lung01 = _resize01(_window_norm(hu_canonicalize(sl_hu), LUNG_MIN, LUNG_MAX))
            medi01 = _resize01(_window_norm(hu_canonicalize(sl_hu), MEDIA_MIN, MEDIA_MAX))
            base01 = medi01 if HF_FROM == "medi" else lung01

            hf_pre = _hf_channel(base01, mode=HF_MODE)
            hf_post = hf_pre * m

            start = C
            panels = [m, hf_pre, hf_post]
            for j in range(3):
                ax = axes[start + j, c_i]
                ax.imshow(panels[j], cmap="gray", vmin=0.0, vmax=1.0)
                ax.set_title(f"{extra_names[j]}\nz={int(z)}", fontsize=10)
                ax.axis("off")

    plt.tight_layout()
    plt.show()


plot_rep_selection_for_patient(\"REDACTED_PATIENT_ID\", n_show=20, seed=42, show_mask_debug=True)


In [ ]:
# Sampling of images for a given fold
def rep_stats_for_fold(fold=None, split="val", seed=42):
    df = dfi.copy()
    if split == "train":
        df = df[df["fold"] != fold]
    else:
        df = df[df["fold"] == fold]

    rows = []
    for _, r in df.iterrows():
        pid = r["patient_id"]
        vol = np.load(r["volume_path"], mmap_mode="r")
        rng = np.random.default_rng(seed + int(pid))
        idx = sample_slice_indices_representative(vol, n_select=N_SLICES_REP, center_fraction=CENTER_FRACTION, rng=rng)
        idx = np.array(sorted(set(map(int, idx))))
        rows.append({
            "patient_id": pid,
            "label": float(r["label"]),
            "label_log": float(r["label_log10"]),
            "Z_original": int(vol.shape[0]),
            "n_rep_selected": int(len(idx)),
            "z_min": int(idx.min()),
            "z_max": int(idx.max()),
            "rep_ratio": float(len(idx) / max(1, vol.shape[0])),
        })
    out = pd.DataFrame(rows).sort_values("patient_id").reset_index(drop=True)
    return out

df_rep = rep_stats_for_fold(fold=1, split="val", seed=42)
display(df_rep)
print("min/median/max n_rep_selected:",
      df_rep["n_rep_selected"].min(),
      df_rep["n_rep_selected"].median(),
      df_rep["n_rep_selected"].max())



In [ ]:
# Cell B — MosMed slice dataset (train/val/test split, representative sampling)
# Updated: optional kp_img + kp_mask outputs for ORB/SIFT-BoVW fusion (no breaking changes)

!pip -q install nibabel

import os
import numpy as np
import pandas as pd
import nibabel as nib
import torch
from torch.utils.data import Dataset
import cv2


class MosMedSliceDatasetSplit(Dataset):
    """
    Slice dataset for MosMed fine-tuning.

    One item = one representative slice:
      x: (C,H,W) from preprocess_slice
      y: float severity in {0,1,2,3,4}
      pid: "study_0001"
      z: slice index in (Z,H,W)

    Optional ORB/SIFT support:
      If return_kp=True, also returns:
        kp_img: (H,W) float in [0,1]  (image used for keypoints/descriptors)
        kp_mask: (H,W) float in [0,1] (lung mask in same space), if return_kp_mask=True

    Notes:
      - split is patient-level (train/val/test) from mosmed_split.csv
      - sampling uses your representative sampling function
      - keypoint extraction must use TRAIN-only vocabulary to avoid leakage
    """
    def __init__(
        self,
        mosmed_split_csv: str,
        split: str,                       # "train" | "val" | "test"
        seed: int = 42,
        n_select: int = N_SLICES_REP,
        center_fraction: float = CENTER_FRACTION,
        expected_channels: int | None = None,
        slice_axis: int = -1,
        max_patients: int | None = None,

        # --- NEW: ORB/SIFT support ---
        return_kp: bool = False,
        return_kp_mask: bool = True,
        keypoint_source: str = "medi",    # {"medi","lung","hf"}
    ):
        assert split in {"train", "val", "test"}
        self.split = split
        self.seed = int(seed)
        self.n_select = int(n_select)
        self.center_fraction = float(center_fraction)
        self.expected_channels = expected_channels
        self.slice_axis = int(slice_axis)

        self.return_kp = bool(return_kp)
        self.return_kp_mask = bool(return_kp_mask)
        self.keypoint_source = str(keypoint_source).lower().strip()
        if self.keypoint_source not in {"medi", "lung", "hf"}:
            raise ValueError("keypoint_source must be one of: {'medi','lung','hf'}")

        df = pd.read_csv(mosmed_split_csv).copy()
        df["patient_id"] = df["patient_id"].astype(str)
        df["split"] = df["split"].astype(str)
        df = df[df["split"] == split].reset_index(drop=True)

        if max_patients is not None:
            pids = sorted(df["patient_id"].unique())
            pids = pids[: int(max_patients)]
            df = df[df["patient_id"].isin(pids)].reset_index(drop=True)

        self.df = df
        self.rows = []  # (pid, y, volume_path, z, Z)

        for pid, g in df.groupby("patient_id", sort=True):
            r0 = g.iloc[0]
            vp = str(r0["volume_path"])
            y = float(r0["severity_ct"])

            vol = self._load_volume(vp)  # (Z,H,W)
            Z = int(vol.shape[0])

            rng = self._make_rng(pid, tag=f"mosmed_{split}")
            idx = sample_slice_indices_representative(
                vol,
                rng=rng,
                center_fraction=self.center_fraction,
                n_select=self.n_select,
            )
            if idx is None or len(idx) == 0:
                idx = np.array([max(0, Z // 2)], dtype=int)

            idx = np.asarray(idx, dtype=int)
            idx = idx[(idx >= 0) & (idx < Z)]
            idx = np.unique(idx)

            for z in idx:
                self.rows.append((str(pid), float(y), vp, int(z), Z))

        if len(self.rows) == 0:
            raise RuntimeError(f"MosMed {split}: produced 0 slices. Check split CSV and sampling params.")

        if self.split == "train":
            rng = np.random.default_rng(self.seed + 1337)
            rng.shuffle(self.rows)

    def _make_rng(self, pid: str, tag: str) -> np.random.Generator:
        s = (hash(f"{pid}|{tag}|seed{self.seed}") & 0xffffffff)
        return np.random.default_rng(s)

    def _load_volume(self, vp: str) -> np.ndarray:
        img = nib.load(vp)
        arr = img.get_fdata(dtype=np.float32)
        arr = np.moveaxis(arr, self.slice_axis, 0)  # -> (Z,H,W)
        arr = np.squeeze(arr)
        if arr.ndim != 3:
            raise ValueError(f"Unexpected MosMed volume shape after moveaxis/squeeze: {arr.shape} for {vp}")
        return arr.astype(np.float32)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i: int):
        pid, y, vp, z, Z = self.rows[i]
        vol = self._load_volume(vp)      # (Z,H,W)
        sl = vol[int(z)]                 # (H,W)

        # CNN input
        x = preprocess_slice(sl)         # (C,H,W)
        x = np.asarray(x, dtype=np.float32)

        if x.ndim == 2:
            x = x[None, :, :]
        elif x.ndim != 3:
            raise ValueError(f"preprocess_slice must return (H,W) or (C,H,W). Got {x.shape}")

        if self.expected_channels is not None and x.shape[0] != int(self.expected_channels):
            raise ValueError(f"Expected {self.expected_channels} channels but got {x.shape[0]}")

        x_t = torch.from_numpy(x).float()
        y_t = torch.tensor(float(y), dtype=torch.float32)

        # Optional keypoint image + mask
        if self.return_kp:
            sl_hu = hu_canonicalize(np.asarray(sl, dtype=np.float32))

            lung01 = _resize01(_window_norm(sl_hu, LUNG_MIN, LUNG_MAX))
            medi01 = _resize01(_window_norm(sl_hu, MEDIA_MIN, MEDIA_MAX))

            if self.keypoint_source == "medi":
                kp_img = medi01
            elif self.keypoint_source == "lung":
                kp_img = lung01
            else:
                # Use current HF operator derived from HF_FROM base
                base01 = medi01 if HF_FROM == "medi" else lung01
                kp_img = _hf_channel(base01, mode=HF_MODE)

            kp_img_t = torch.from_numpy(np.asarray(kp_img, dtype=np.float32)).float()

            if self.return_kp_mask:
                m = _lung_mask_from_hu(sl_hu)  # (H,W) {0,1} in HU space
                m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)
                m = np.clip(m, 0.0, 1.0)
                if float(LUNG_MASK_EDGE_BLUR_SIGMA) > 0:
                    m = cv2.GaussianBlur(
                        m, (0, 0),
                        sigmaX=float(LUNG_MASK_EDGE_BLUR_SIGMA),
                        sigmaY=float(LUNG_MASK_EDGE_BLUR_SIGMA),
                    )
                    m = np.clip(m, 0.0, 1.0)
                kp_mask_t = torch.from_numpy(m).float()
                return x_t, y_t, pid, int(z), kp_img_t, kp_mask_t

            return x_t, y_t, pid, int(z), kp_img_t

        # default (unchanged)
        return x_t, y_t, pid, int(z)


# sanity checks
PROJECT_ROOT = DATA_ROOT
MOSMED_SPLIT = os.path.join(PROJECT_ROOT, "external_data/mosmed/mosmed_split.csv")

ds_tr = MosMedSliceDatasetSplit(MOSMED_SPLIT, split="train", expected_channels=3)
ds_va = MosMedSliceDatasetSplit(MOSMED_SPLIT, split="val", expected_channels=3)
ds_te = MosMedSliceDatasetSplit(MOSMED_SPLIT, split="test", expected_channels=3)

print("MosMed slices | train:", len(ds_tr), "| val:", len(ds_va), "| test:", len(ds_te))
x, y, pid, z = ds_tr[0]
print("Example:", pid, "z=", z, "x:", tuple(x.shape), "y=", float(y))

# kp sanity check
ds_kp = MosMedSliceDatasetSplit(
    MOSMED_SPLIT, split="train", expected_channels=3,
    return_kp=True, return_kp_mask=True, keypoint_source="medi"
)
x2, y2, pid2, z2, kp_img, kp_mask = ds_kp[0]
print("KP example:", pid2, "z=", z2, "kp_img:", tuple(kp_img.shape), "kp_mask:", tuple(kp_mask.shape))


In [ ]:
# Cell 09A — Slice-level Dataset (representative sampling) for weakly-supervised training


!pip -q install torch torchvision

import torch
from torch.utils.data import Dataset
import pandas as pd
import numpy as np
import cv2


class SliceDatasetRepresentative(Dataset):
    def __init__(
        self,
        index_csv: str,
        fold: int,
        split: str,
        seed: int = 42,
        center_fraction: float = CENTER_FRACTION,
        n_select: int = N_SLICES_REP,
        return_z_norm: bool = False,
        expected_channels: int | None = None,
        return_kp: bool = False,
        return_kp_mask: bool = True,
        keypoint_source: str = "medi",   # {"medi","lung","hf"} - used to build kp_img in [0,1]
    ):
        """
        Slice dataset with patient-level weak labels and representative slice sampling.

        Returns per __getitem__ (base):
            x: torch.FloatTensor, shape (C,H,W)
            y: torch.FloatTensor, scalar (patient label, weak supervision)
            pid: str
            z: int (slice index)
            z_norm: float in [0,1] optional

        If return_kp=True, also returns:
            kp_img: torch.FloatTensor, shape (H,W) in [0,1], used as input to ORB/SIFT
            kp_mask: torch.FloatTensor, shape (H,W) in [0,1] (if return_kp_mask=True)
        """
        assert split in ["train", "val"]
        df = pd.read_csv(index_csv)
        df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)
        df["label"] = df["label_log10"].astype(float)
        df["fold"] = df["fold"].astype(int)

        if split == "train":
            self.df = df[df["fold"] != fold].reset_index(drop=True)
        else:
            self.df = df[df["fold"] == fold].reset_index(drop=True)

        self.split = split
        self.fold = int(fold)
        self.seed = int(seed)

        self.center_fraction = float(center_fraction)
        self.n_select = int(n_select)
        self.return_z_norm = bool(return_z_norm)
        self.expected_channels = expected_channels

        self.return_kp = bool(return_kp)
        self.return_kp_mask = bool(return_kp_mask)
        self.keypoint_source = str(keypoint_source).lower().strip()
        if self.keypoint_source not in {"medi", "lung", "hf"}:
            raise ValueError("keypoint_source must be one of: {'medi','lung','hf'}")

        # Flat "slice index" table: one row per selected slice across all patients
        self.rep_cache: dict[str, np.ndarray] = {}  # pid -> selected z indices
        self.rows: list[tuple[str, float, str, int, int]] = []  # (pid, y, volume_path, z, Z)

        for _, r in self.df.iterrows():
            pid = str(r["patient_id"]).zfill(4)
            y = float(r["label"])
            vp = str(r["volume_path"])

            vol = np.load(vp, mmap_mode="r")  # (Z,H,W)
            Z = int(vol.shape[0])

            rng = self._make_deterministic_rng(pid, "rep_train" if split == "train" else "rep_val")
            idx = sample_slice_indices_representative(
                vol,
                rng=rng,
                center_fraction=self.center_fraction,
                n_select=self.n_select
            )

            if idx is None or len(idx) == 0:
                idx = np.array([max(0, Z // 2)], dtype=int)

            idx = np.asarray(idx, dtype=int)
            idx = idx[(idx >= 0) & (idx < Z)]
            idx = np.unique(idx)

            self.rep_cache[pid] = idx

            for z in idx:
                self.rows.append((pid, y, vp, int(z), Z))

        if len(self.rows) == 0:
            raise RuntimeError("No slices found. Check representative sampling and volume paths.")

        # Training: shuffle slice order deterministically to avoid patient-block ordering
        if self.split == "train":
            rng = np.random.default_rng(self.seed + 1337 + self.fold)
            rng.shuffle(self.rows)

    def _make_deterministic_rng(self, pid: str, tag: str) -> np.random.Generator:
        s = (hash(f"{pid}|fold{self.fold}|{tag}|seed{self.seed}") & 0xffffffff)
        return np.random.default_rng(s)

    def __len__(self) -> int:
        return len(self.rows)

    def __getitem__(self, i: int):
        pid, y, vp, z, Z = self.rows[i]

        vol = np.load(vp, mmap_mode="r")   # (Z,H,W) HU
        sl = vol[int(z)]                   # (H,W)

        # CNN input (C,H,W)
        x = preprocess_slice(sl)
        x = np.asarray(x)

        if x.ndim == 2:
            x = x[None, :, :]
        elif x.ndim != 3:
            raise ValueError(f"preprocess_slice must return (H,W) or (C,H,W). Got shape={x.shape}")

        if self.expected_channels is not None and x.shape[0] != int(self.expected_channels):
            raise ValueError(
                f"Expected {self.expected_channels} channels but got {x.shape[0]}. "
                "Check preprocess_slice / ADD_HF_CHANNEL / HF_MODE settings."
            )

        x_t = torch.from_numpy(x).float()
        y_t = torch.tensor(float(y), dtype=torch.float32)

        z_norm = 0.0 if Z <= 1 else float(z) / float(Z - 1)
        z_norm_t = torch.tensor(z_norm, dtype=torch.float32)

        # keypoint image + mask for ORB/SIFT
        if self.return_kp:
            sl_hu = hu_canonicalize(np.asarray(sl, dtype=np.float32))

            lung01 = _resize01(_window_norm(sl_hu, LUNG_MIN, LUNG_MAX))
            medi01 = _resize01(_window_norm(sl_hu, MEDIA_MIN, MEDIA_MAX))

            if self.keypoint_source == "medi":
                kp_img = medi01
            elif self.keypoint_source == "lung":
                kp_img = lung01
            else:
                base01 = medi01 if HF_FROM == "medi" else lung01
                kp_img = _hf_channel(base01, mode=HF_MODE)

            kp_img_t = torch.from_numpy(np.asarray(kp_img, dtype=np.float32)).float()

            if self.return_kp_mask:
                m = _lung_mask_from_hu(sl_hu)  # (H,W) {0,1}
                m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32)
                m = np.clip(m, 0.0, 1.0)

                if float(LUNG_MASK_EDGE_BLUR_SIGMA) > 0:
                    m = cv2.GaussianBlur(
                        m, (0, 0),
                        sigmaX=float(LUNG_MASK_EDGE_BLUR_SIGMA),
                        sigmaY=float(LUNG_MASK_EDGE_BLUR_SIGMA)
                    )
                    m = np.clip(m, 0.0, 1.0)

                kp_mask_t = torch.from_numpy(m).float()
            else:
                kp_mask_t = None

        # return tuples (backward compatible)
        if self.return_z_norm and self.return_kp:
            if self.return_kp_mask:
                return x_t, y_t, pid, int(z), z_norm_t, kp_img_t, kp_mask_t
            return x_t, y_t, pid, int(z), z_norm_t, kp_img_t

        if self.return_z_norm:
            return x_t, y_t, pid, int(z), z_norm_t

        if self.return_kp:
            if self.return_kp_mask:
                return x_t, y_t, pid, int(z), kp_img_t, kp_mask_t
            return x_t, y_t, pid, int(z), kp_img_t

        return x_t, y_t, pid, int(z)




# quick sanity + size prints

TRAIN_FOLD_FOR_PRINT = 0

ds = SliceDatasetRepresentative(
    index_csv=f"{OUT_ROOT}/train_index.csv",
    fold=TRAIN_FOLD_FOR_PRINT,
    split="train",
    seed=42,
    expected_channels=3,
)
ds_val = SliceDatasetRepresentative(
    index_csv=f"{OUT_ROOT}/train_index.csv",
    fold=TRAIN_FOLD_FOR_PRINT,
    split="val",
    seed=42,
    expected_channels=3,
)

print(f"[Fold {TRAIN_FOLD_FOR_PRINT}] train patients: {len(ds.df)} | val patients: {len(ds_val.df)}")
print(f"[Fold {TRAIN_FOLD_FOR_PRINT}] train slices:   {len(ds)} | val slices:   {len(ds_val)}")
try:
    _p_tr = [r[0] for r in ds.rows]
    _p_va = [r[0] for r in ds_val.rows]
    _u_tr, _c_tr = np.unique(_p_tr, return_counts=True)
    _u_va, _c_va = np.unique(_p_va, return_counts=True)
    print(f"[Fold {TRAIN_FOLD_FOR_PRINT}] slices/patient train: mean={_c_tr.mean():.1f}, min={_c_tr.min()}, max={_c_tr.max()}")
    print(f"[Fold {TRAIN_FOLD_FOR_PRINT}] slices/patient val:   mean={_c_va.mean():.1f}, min={_c_va.min()}, max={_c_va.max()}")
except Exception as e:
    print("[warn] could not compute slices/patient stats:", str(e))

# example item
x, y, pid, z = ds[0]
print("Train slice example:", pid, "z=", z, "x:", tuple(x.shape), "y:", float(y))

# kp outputs sanity check
ds_kp = SliceDatasetRepresentative(
    index_csv=f"{OUT_ROOT}/train_index.csv",
    fold=TRAIN_FOLD_FOR_PRINT,
    split="train",
    seed=42,
    expected_channels=3,
    return_kp=True,
    return_kp_mask=True,
    keypoint_source="medi",
)
x3, y3, pid3, z3, kp_img, kp_mask = ds_kp[0]
print("Train slice example (kp):",
      pid3, "z=", z3, "x:",
      tuple(x3.shape), "kp_img:",
      tuple(kp_img.shape), "kp_mask:",
      tuple(kp_mask.shape))



In [ ]:
# Cell 09B — MosMed slice dataset (external validation; ordinal labels CT-0..CT-4)

!pip -q install nibabel

import os
import numpy as np
import pandas as pd
import nibabel as nib
import torch
from torch.utils.data import Dataset


class MosMedSliceDatasetRepresentative(Dataset):
    """
    Inference-only dataset for MosMed external validation.

    Each __getitem__ returns a single representative slice from a patient volume:
      x: (C,H,W) float32 in [0,1]  from preprocess_slice
      y: float32 severity grade in {0,1,2,3,4}
      pid: str (e.g., "study_0001")
      z: int slice index within (Z,H,W)
    """
    def __init__(
        self,
        mosmed_index_csv: str,
        seed: int = 42,
        n_select: int = N_SLICES_REP,
        center_fraction: float = CENTER_FRACTION,
        expected_channels: int | None = None,
        slice_axis: int = -1,  # NIfTI usually stores slices on last axis; we default to that.
    ):
        self.df = pd.read_csv(mosmed_index_csv).copy()
        self.df["patient_id"] = self.df["patient_id"].astype(str)
        self.df["severity_ct"] = self.df["severity_ct"].astype(int)
        self.df["volume_path"] = self.df["volume_path"].astype(str)

        self.seed = int(seed)
        self.n_select = int(n_select)
        self.center_fraction = float(center_fraction)
        self.expected_channels = expected_channels
        self.slice_axis = int(slice_axis)

        # Flat list of rows: one row per selected slice across all patients
        self.rows = []  # (pid, y, volume_path, z, Z)

        for _, r in self.df.iterrows():
            pid = str(r["patient_id"])
            y = int(r["severity_ct"])
            vp = str(r["volume_path"])

            # Load volume header and data
            vol = self._load_volume(vp)         # (Z,H,W) float32-ish
            Z = int(vol.shape[0])

            rng = self._make_rng(pid)
            idx = sample_slice_indices_representative(
                vol,
                rng=rng,
                center_fraction=self.center_fraction,
                n_select=self.n_select,
            )
            if idx is None or len(idx) == 0:
                idx = np.array([max(0, Z // 2)], dtype=int)

            idx = np.asarray(idx, dtype=int)
            idx = idx[(idx >= 0) & (idx < Z)]
            idx = np.unique(idx)

            for z in idx:
                self.rows.append((pid, float(y), vp, int(z), Z))

        if len(self.rows) == 0:
            raise RuntimeError("MosMed dataset produced 0 slices. Check index CSV and sampling.")

    def _make_rng(self, pid: str) -> np.random.Generator:
        s = (hash(f"mosmed|{pid}|seed{self.seed}") & 0xffffffff)
        return np.random.default_rng(s)

    def _load_volume(self, vp: str) -> np.ndarray:
        """
        Load NIfTI and return volume as (Z,H,W) float32.
        We assume slice axis is last axis by default and move it to front.
        """
        img = nib.load(vp)
        arr = img.get_fdata(dtype=np.float32)  # float32

        # Move chosen slice axis to front -> (Z,*,*)
        # If slice_axis=-1 and arr is (H,W,Z), result becomes (Z,H,W)
        arr = np.moveaxis(arr, self.slice_axis, 0)

        # If it is (Z,H,W) now, great. If it has extra dims (rare), squeeze.
        arr = np.asarray(arr)
        if arr.ndim != 3:
            arr = np.squeeze(arr)
        if arr.ndim != 3:
            raise ValueError(f"Unexpected MosMed volume shape after squeeze/moveaxis: {arr.shape} for {vp}")

        return arr.astype(np.float32)

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i: int):
        pid, y, vp, z, Z = self.rows[i]

        vol = self._load_volume(vp)   # (Z,H,W)
        sl = vol[int(z)]              # (H,W)

        x = preprocess_slice(sl)      # (C,H,W)
        x = np.asarray(x, dtype=np.float32)

        if x.ndim == 2:
            x = x[None, :, :]
        elif x.ndim != 3:
            raise ValueError(f"preprocess_slice must return (H,W) or (C,H,W). Got {x.shape}")

        if self.expected_channels is not None and x.shape[0] != int(self.expected_channels):
            raise ValueError(
                f"Expected {self.expected_channels} channels but got {x.shape[0]}. "
                "Check ADD_HF_CHANNEL / HF_MODE / preprocess_slice."
            )

        x_t = torch.from_numpy(x).float()
        y_t = torch.tensor(float(y), dtype=torch.float32)
        return x_t, y_t, pid, int(z)


# quick sanity check
MOSMED_INDEX = "/path/to/authorised/data/external_data/mosmed/mosmed_index.csv"
ds_mos = MosMedSliceDatasetRepresentative(
    mosmed_index_csv=MOSMED_INDEX,
    seed=42,
    n_select=N_SLICES_REP,
    center_fraction=CENTER_FRACTION,
    expected_channels=3,
    slice_axis=-1,
)

x, y, pid, z = ds_mos[0]
print("MosMed slice example:", pid, "z=", z, "x:", tuple(x.shape), "y(severity)=", float(y))
print("Total MosMed slices:", len(ds_mos))


In [ ]:
# Cell 10 — Slice model (flexible input channels + optional handcrafted feature fusion)
# Updated: adds robust loading of TRAINED checkpoints (not RadImageNet) for external validation.

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models


def _clean_state_dict(sd):
    """Original helper: cleans a dict with optional 'state_dict' key + strips DataParallel 'module.' prefixes."""
    if isinstance(sd, dict) and "state_dict" in sd:
        sd = sd["state_dict"]
    out = {}
    for k, v in sd.items():
        if k.startswith("module."):
            k = k[len("module."):]
        out[k] = v
    return out


def _strip_prefix(sd, prefix: str):
    out = {}
    for k, v in sd.items():
        if k.startswith(prefix):
            out[k[len(prefix):]] = v
        else:
            out[k] = v
    return out


def _remap_backbone_keys_for_torchvision(sd, backbone_name: str):
    """
    Fix common naming mismatches between external checkpoints and torchvision models.
    """
    b = backbone_name.lower().strip()
    if b == "densenet121":
        if any(k.startswith("0.") for k in sd.keys()):
            sd = {("features." + k[2:]) if k.startswith("0.") else k: v for k, v in sd.items()}
        sd = {k: v for k, v in sd.items() if not (k.startswith("1.") or k.startswith("classifier."))}
        return sd
    return sd


# -------------------------
# NEW: robust trained checkpoint loading (for your own fold checkpoints)
# -------------------------

def _clean_state_dict_general(ckpt: dict) -> dict:
    """
    Handle common checkpoint formats:
      - raw state_dict (mapping param_name -> tensor)
      - {'state_dict': ...}
      - {'model_state_dict': ...}
      - {'model': ...}
    Also strips DataParallel 'module.' prefixes.
    """
    if not isinstance(ckpt, dict):
        raise ValueError("Checkpoint must be a dict (loaded via torch.load).")

    # Pick likely key if present
    for key in ["state_dict", "model_state_dict", "model", "net", "weights"]:
        if key in ckpt and isinstance(ckpt[key], dict):
            sd = ckpt[key]
            break
    else:
        # assume ckpt itself is a state_dict
        sd = ckpt

    out = {}
    for k, v in sd.items():
        if k.startswith("module."):
            k = k[len("module."):]
        out[k] = v
    return out


def load_trained_checkpoint_into_model(model: nn.Module, ckpt_path: str, strict: bool = True):
    """
    Load a trained checkpoint into a model instance (for inference/external validation).

    Args:
        model: instantiated SliceRegressor with matching architecture.
        ckpt_path: path to torch checkpoint (.pt/.pth).
        strict: True for exact match (recommended for true external validation).

    Returns:
        (missing_keys, unexpected_keys) from load_state_dict.
    """
    ckpt = torch.load(ckpt_path, map_location="cpu")
    sd = _clean_state_dict_general(ckpt)

    # Optional: handle if user saved keys under "model." or "backbone."
    # Keep this conservative: only strip if it clearly matches.
    if any(k.startswith("model.") for k in sd.keys()):
        sd = _strip_prefix(sd, "model.")
    if any(k.startswith("backbone.") for k in sd.keys()):
        # If checkpoint stored as backbone.<...>, map it to model.backbone.<...>?
        # Here we assume the checkpoint matches the full model unless told otherwise.
        pass

    missing, unexpected = model.load_state_dict(sd, strict=strict)
    print(f"[ckpt] loaded: {ckpt_path}")
    print("  strict:", strict)
    print("  missing keys (first 15):", list(missing)[:15])
    print("  unexpected keys (first 15):", list(unexpected)[:15])
    return missing, unexpected


# -------------------------
# Stems (map in_ch -> 3 so pretrained backbones can be used unchanged)
# -------------------------

class DepthwiseSeparableStem(nn.Module):
    def __init__(self, in_ch: int, out_ch: int = 3, k: int = 5):
        super().__init__()
        assert in_ch >= 1, "in_ch must be >= 1"
        pad = k // 2
        self.dw = nn.Conv2d(in_ch, in_ch, kernel_size=k, padding=pad, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.dw(x)
        x = self.pw(x)
        x = self.bn(x)
        return self.act(x)


class Conv1x1Then5x5Stem(nn.Module):
    def __init__(self, in_ch: int, out_ch: int = 3):
        super().__init__()
        self.proj = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.conv = nn.Conv2d(out_ch, out_ch, kernel_size=5, padding=2, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        x = self.proj(x)
        x = self.conv(x)
        x = self.bn(x)
        return self.act(x)


# -------------------------
# Optional attention blocks
# -------------------------

class SEBlock(nn.Module):
    """Squeeze-and-Excitation on feature vector [B,D]."""
    def __init__(self, dim, r=8):
        super().__init__()
        h = max(8, dim // r)
        self.fc1 = nn.Linear(dim, h)
        self.fc2 = nn.Linear(h, dim)

    def forward(self, f):
        s = torch.sigmoid(self.fc2(F.relu(self.fc1(f))))
        return f * s


class FeatureMHSA(nn.Module):
    def __init__(self, feat_dim: int, tokens: int = 8, token_dim: int = 64, heads: int = 4, dropout: float = 0.1):
        super().__init__()
        assert tokens >= 2
        assert token_dim % heads == 0
        self.tokens = int(tokens)
        self.token_dim = int(token_dim)

        self.to_tokens = nn.Linear(feat_dim, self.tokens * self.token_dim)
        self.mha = nn.MultiheadAttention(embed_dim=self.token_dim, num_heads=heads, dropout=dropout, batch_first=True)
        self.ln = nn.LayerNorm(self.token_dim)
        self.back = nn.Linear(self.tokens * self.token_dim, feat_dim)

    def forward(self, f):
        B = f.shape[0]
        t = self.to_tokens(f).view(B, self.tokens, self.token_dim)
        att, _ = self.mha(t, t, t, need_weights=False)
        t = self.ln(t + att)
        t = t.reshape(B, self.tokens * self.token_dim)
        return self.back(t)


# -------------------------
# Model
# -------------------------

class SliceRegressor(nn.Module):
    def __init__(
        self,
        backbone: str = "resnet18",
        in_ch: int = 3,
        imagenet_pretrained: bool = True,
        rad_weight_path: str | None = None,
        conv_stem: str | None = None,     # None | "1x1_5x5" | "dwsep"
        attention: str | None = None,     # None | "se" | "mhsa"
        mhsa_tokens: int = 8,
        mhsa_token_dim: int = 64,
        mhsa_heads: int = 4,

        # handcrafted feature fusion
        handcrafted_dim: int | None = None,
        handcrafted_proj: int = 64,
        fusion: str = "concat",
        head_dropout: float = 0.2,
    ):
        super().__init__()
        self.backbone_name = backbone.lower().strip()
        self.in_ch = int(in_ch)

        # input stem
        if self.in_ch == 3:
            self.in_proj = nn.Identity()
        else:
            if conv_stem is None:
                self.in_proj = nn.Conv2d(self.in_ch, 3, kernel_size=1, bias=False)
            elif conv_stem == "1x1_5x5":
                self.in_proj = Conv1x1Then5x5Stem(in_ch=self.in_ch, out_ch=3)
            elif conv_stem == "dwsep":
                self.in_proj = DepthwiseSeparableStem(in_ch=self.in_ch, out_ch=3, k=5)
            else:
                raise ValueError("conv_stem must be None | '1x1_5x5' | 'dwsep'")

        # backbone selection
        if self.backbone_name == "resnet18":
            net = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if imagenet_pretrained else None)
            feat_dim = net.fc.in_features
            net.fc = nn.Identity()
            self.backbone = net

        elif self.backbone_name == "resnet50":
            net = models.resnet50(weights=models.ResNet50_Weights.DEFAULT if imagenet_pretrained else None)
            feat_dim = net.fc.in_features
            net.fc = nn.Identity()
            self.backbone = net

        elif self.backbone_name == "resnet101":
            net = models.resnet101(weights=models.ResNet101_Weights.DEFAULT if imagenet_pretrained else None)
            feat_dim = net.fc.in_features
            net.fc = nn.Identity()
            self.backbone = net

        elif self.backbone_name == "densenet121":
            net = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT if imagenet_pretrained else None)
            feat_dim = net.classifier.in_features
            net.classifier = nn.Identity()
            self.backbone = net

        elif self.backbone_name == "inceptionv3":
            net = models.inception_v3(
                weights=models.Inception_V3_Weights.DEFAULT if imagenet_pretrained else None,
                aux_logits=False,
                init_weights=True
            )
            self.backbone = nn.Sequential(
                net.Conv2d_1a_3x3, net.Conv2d_2a_3x3, net.Conv2d_2b_3x3, net.maxpool1,
                net.Conv2d_3b_1x1, net.Conv2d_4a_3x3, net.maxpool2,
                net.Mixed_5b, net.Mixed_5c, net.Mixed_5d,
                net.Mixed_6a, net.Mixed_6b, net.Mixed_6c, net.Mixed_6d, net.Mixed_6e,
                net.Mixed_7a, net.Mixed_7b, net.Mixed_7c,
                net.avgpool, nn.Flatten(1),
            )
            feat_dim = 2048
        else:
            raise ValueError(f"Unsupported backbone: {backbone}")

        self.feat_norm = nn.LayerNorm(feat_dim)

        if attention is None:
            self.att = None
        elif attention == "se":
            self.att = SEBlock(feat_dim, r=8)
        elif attention == "mhsa":
            self.att = FeatureMHSA(
                feat_dim=feat_dim,
                tokens=mhsa_tokens,
                token_dim=mhsa_token_dim,
                heads=mhsa_heads,
                dropout=0.1
            )
        else:
            raise ValueError("attention must be None | 'se' | 'mhsa'")

        self.fusion = str(fusion).lower().strip()
        if self.fusion not in {"concat"}:
            raise ValueError("fusion must be 'concat' (for now)")

        self.handcrafted_dim = None if handcrafted_dim is None else int(handcrafted_dim)
        if self.handcrafted_dim is None:
            self.hand_proj = None
            fused_dim = feat_dim
        else:
            proj_dim = int(handcrafted_proj)
            if proj_dim <= 0:
                raise ValueError("handcrafted_proj must be > 0")
            self.hand_proj = nn.Sequential(
                nn.LayerNorm(self.handcrafted_dim),
                nn.Linear(self.handcrafted_dim, proj_dim),
                nn.ReLU(inplace=True),
            )
            fused_dim = feat_dim + proj_dim

        self.head = nn.Sequential(
            nn.Linear(fused_dim, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(float(head_dropout)),
            nn.Linear(128, 1)
        )

        # load RadImageNet weights (optional)
        if rad_weight_path is not None:
            ckpt = torch.load(rad_weight_path, map_location="cpu")
            sd = _clean_state_dict(ckpt)
            sd = _strip_prefix(sd, "backbone.")
            sd = _strip_prefix(sd, "model.")
            sd = {k: v for k, v in sd.items() if not k.startswith("AuxLogits.")}
            sd = _remap_backbone_keys_for_torchvision(sd, self.backbone_name)

            missing, unexpected = self.backbone.load_state_dict(sd, strict=False)
            print(f"[RadImageNet] loading from: {rad_weight_path}")
            print("  missing keys (first 10):", list(missing)[:10])
            print("  unexpected keys (first 10):", list(unexpected)[:10])

    def forward(self, x, handcrafted: torch.Tensor | None = None):
        x = self.in_proj(x)

        if self.backbone_name == "inceptionv3" and x.shape[-1] != 299:
            x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)

        f = self.backbone(x)
        f = torch.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0)
        f = self.feat_norm(f)

        if self.att is not None:
            f = self.att(f)

        if self.hand_proj is not None:
            if handcrafted is None:
                raise ValueError("Model was created with handcrafted_dim but handcrafted=None was passed to forward().")
            h = self.hand_proj(handcrafted)
            f = torch.cat([f, h], dim=1)

        return self.head(f).squeeze(-1)

    forward_slice = forward



In [ ]:
# Cell 11 — Patient-level evaluation from slice loader (LOG + RAW metrics)
# Updated: adds ordinal external-validation evaluation (e.g., MosMed CT-0..CT-4)

import numpy as np
import torch
from collections import defaultdict
from scipy.stats import spearmanr, pearsonr


def aggregate_scores(scores, mode="mean", topk_frac=0.2, topk_k=None, mix=0.7):
    scores = np.asarray(scores, dtype=float)
    if len(scores) == 0:
        return 0.0

    if mode == "mean":
        return float(scores.mean())

    if topk_k is not None:
        k = int(topk_k)
    else:
        k = int(max(1, int(len(scores) * float(topk_frac))))
    k = max(1, min(k, len(scores)))

    top = float(np.sort(scores)[-k:].mean())
    if mode == "topk":
        return top
    if mode == "mix":
        return float(mix * top + (1.0 - mix) * scores.mean())

    raise ValueError("mode must be mean/topk/mix")


def _regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

    sse = float(np.sum((y_true - y_pred) ** 2))
    sst = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - sse / (sst + 1e-12))

    if np.std(y_true) < 1e-12 or np.std(y_pred) < 1e-12:
        rho, r = 0.0, 0.0
    else:
        rho = spearmanr(y_true, y_pred).correlation
        rho = 0.0 if (rho is None or np.isnan(rho)) else float(rho)
        r = pearsonr(y_true, y_pred)[0]
        r = 0.0 if (r is None or np.isnan(r)) else float(r)

    return {"MAE": mae, "RMSE": rmse, "R2": r2, "Spearman": rho, "Pearson": r}


def eval_patient_level_from_slices(
    model,
    loader,
    device,
    agg_mode="topk",
    topk_frac=0.2,
    topk_k=None,
    mix=0.7,
    compute_raw_metrics=True,
    use_handcrafted: bool = False,
    bovw_cache: dict | None = None,
    bovw_k: int | None = None,
):
    """
    INTERNAL evaluation (your main task):
      - model outputs LOG10 predictions
      - metrics in LOG space
      - optional RAW metrics via raw = 10**log

    If use_handcrafted=True, uses bovw_cache[(pid,z)] vectors passed to model.
    """
    model.eval()
    preds_by_pid = defaultdict(list)
    y_by_pid = {}

    if use_handcrafted:
        if bovw_cache is None or bovw_k is None:
            raise ValueError("use_handcrafted=True requires bovw_cache and bovw_k.")
        K = int(bovw_k)

    with torch.no_grad():
        for batch in loader:
            x, y, pid, z = batch[0], batch[1], batch[2], batch[3]
            x = x.to(device)

            if use_handcrafted:
                feats = []
                for i_b in range(len(z)):
                    key = (str(pid[i_b]), int(z[i_b]))
                    f = bovw_cache.get(key, None)
                    if f is None:
                        f = np.zeros((K,), dtype=np.float32)
                    feats.append(f)
                bovw = torch.from_numpy(np.stack(feats, axis=0)).float().to(device)
                preds = model(x, handcrafted=bovw).detach().cpu().numpy().astype(float)
            else:
                preds = model(x).detach().cpu().numpy().astype(float)

            y_np = y.detach().cpu().numpy().astype(float)

            for i in range(len(preds)):
                p = str(pid[i])
                preds_by_pid[p].append(float(preds[i]))
                y_by_pid[p] = float(y_np[i])

    pids = sorted(preds_by_pid.keys())
    y_true_log = np.array([y_by_pid[p] for p in pids], dtype=float)
    y_pred_log = np.array(
        [aggregate_scores(preds_by_pid[p], mode=agg_mode, topk_frac=topk_frac, topk_k=topk_k, mix=mix) for p in pids],
        dtype=float,
    )

    out = {}
    log_metrics = _regression_metrics(y_true_log, y_pred_log)
    out.update({f"{k}_log": v for k, v in log_metrics.items()})

    if compute_raw_metrics:
        y_true_raw = np.power(10.0, y_true_log)
        y_pred_raw = np.power(10.0, y_pred_log)
        raw_metrics = _regression_metrics(y_true_raw, y_pred_raw)
        out.update({f"{k}_raw": v for k, v in raw_metrics.items()})

    return out


# -------------------------------
# ordinal external validation
# -------------------------------

def _ordinal_metrics(y_true_ord, y_pred_cont, use_numeric_map: bool = True, numeric_map=None):
    """
    Ordinal evaluation for external datasets (e.g., MosMed CT-0..CT-4).

    Always returns Spearman (primary).

    Optionally returns MAE/RMSE after mapping ordinal labels to numeric values.
    Default numeric_map corresponds to midpoints of involvement buckets:
      0->0, 1->12.5, 2->37.5, 3->62.5, 4->87.5
    """
    y_true_ord = np.asarray(y_true_ord, dtype=float)
    y_pred_cont = np.asarray(y_pred_cont, dtype=float)

    # Spearman on ordinal scale (most meaningful)
    if np.std(y_true_ord) < 1e-12 or np.std(y_pred_cont) < 1e-12:
        rho = 0.0
    else:
        rho = spearmanr(y_true_ord, y_pred_cont).correlation
        rho = 0.0 if (rho is None or np.isnan(rho)) else float(rho)

    out = {"Spearman": rho}

    if use_numeric_map:
        if numeric_map is None:
            numeric_map = {0: 0.0, 1: 12.5, 2: 37.5, 3: 62.5, 4: 87.5}
        y_true_num = np.array([numeric_map.get(int(v), float(v)) for v in y_true_ord], dtype=float)
        mae = float(np.mean(np.abs(y_true_num - y_pred_cont)))
        rmse = float(np.sqrt(np.mean((y_true_num - y_pred_cont) ** 2)))
        out.update({"MAE_num": mae, "RMSE_num": rmse})

    return out


def eval_patient_level_ordinal_from_slices(
    model,
    loader,
    device,
    agg_mode="topk",
    topk_frac=0.2,
    topk_k=None,
    mix=0.7,
    use_handcrafted: bool = False,
    bovw_cache: dict | None = None,
    bovw_k: int | None = None,
    use_numeric_map: bool = False,
    numeric_map=None,
):
    """
    EXTERNAL evaluation (e.g., MosMed):
      - y is an ordinal grade (0..4)
      - model outputs continuous scores
      - aggregate per patient, then compute Spearman (primary)
      - optionally compute MAE/RMSE against a numeric mapping of the ordinal grades

    If use_handcrafted=True, uses bovw_cache[(pid,z)] vectors passed to model.
    """
    model.eval()
    preds_by_pid = defaultdict(list)
    y_by_pid = {}

    if use_handcrafted:
        if bovw_cache is None or bovw_k is None:
            raise ValueError("use_handcrafted=True requires bovw_cache and bovw_k.")
        K = int(bovw_k)

    with torch.no_grad():
        for batch in loader:
            x, y, pid, z = batch[0], batch[1], batch[2], batch[3]
            x = x.to(device)

            if use_handcrafted:
                feats = []
                for i_b in range(len(z)):
                    key = (str(pid[i_b]), int(z[i_b]))
                    f = bovw_cache.get(key, None)
                    if f is None:
                        f = np.zeros((K,), dtype=np.float32)
                    feats.append(f)
                bovw = torch.from_numpy(np.stack(feats, axis=0)).float().to(device)
                preds = model(x, handcrafted=bovw).detach().cpu().numpy().astype(float)
            else:
                preds = model(x).detach().cpu().numpy().astype(float)

            y_np = y.detach().cpu().numpy().astype(float)

            for i in range(len(preds)):
                p = str(pid[i])
                preds_by_pid[p].append(float(preds[i]))
                y_by_pid[p] = float(y_np[i])  # ordinal label per patient (repeated per slice)

    pids = sorted(preds_by_pid.keys())
    y_true_ord = np.array([y_by_pid[p] for p in pids], dtype=float)
    y_pred = np.array(
        [aggregate_scores(preds_by_pid[p], mode=agg_mode, topk_frac=topk_frac, topk_k=topk_k, mix=mix) for p in pids],
        dtype=float,
    )

    return _ordinal_metrics(y_true_ord, y_pred, use_numeric_map=use_numeric_map, numeric_map=numeric_map)




In [ ]:
# Cell 12 — Slice-level training + patient-level validation early stopping
# ORB -> BoVW handcrafted features + fusion with CNN embedding

import torch
import numpy as np
from torch.utils.data import DataLoader
import torchvision.transforms.functional as TF
import cv2


def augment_ct_batch(x, max_deg=5, max_shift=6, jitter=0.03, noise=0.02):
    B = x.shape[0]
    out = []
    for b in range(B):
        xb = x[b]
        angle = float(np.random.uniform(-max_deg, max_deg))
        tx = int(np.random.uniform(-max_shift, max_shift))
        ty = int(np.random.uniform(-max_shift, max_shift))
        scale = float(np.random.uniform(0.98, 1.02))

        chs = []
        for c in range(xb.shape[0]):
            img = xb[c:c+1]
            img = TF.affine(img, angle=angle, translate=[tx, ty], scale=scale, shear=[0.0, 0.0])
            chs.append(img)
        xb = torch.cat(chs, dim=0)

        xb = torch.clamp(
            xb * float(np.random.uniform(1.0 - jitter, 1.0 + jitter)) + float(np.random.uniform(-jitter, jitter)),
            0.0, 1.0
        )
        xb = torch.clamp(xb + noise * torch.randn_like(xb), 0.0, 1.0)
        out.append(xb)

    return torch.stack(out, dim=0)


def set_backbone_trainable(model, trainable: bool):
    for p in model.backbone.parameters():
        p.requires_grad = trainable


def set_partial_trainable(model, trainable: bool):
    bb = model.backbone

    if hasattr(bb, "encoder") and hasattr(bb.encoder, "layers"):
        layers = bb.encoder.layers
        for p in layers[-2:].parameters():
            p.requires_grad = trainable
        return

    if hasattr(bb, "layer4"):
        for p in bb.layer4.parameters():
            p.requires_grad = trainable
        return

    if hasattr(bb, "features") and hasattr(bb.features, "denseblock4"):
        for p in bb.features.denseblock4.parameters():
            p.requires_grad = trainable
        if hasattr(bb.features, "norm5"):
            for p in bb.features.norm5.parameters():
                p.requires_grad = trainable
        return

    if isinstance(bb, torch.nn.Sequential) and len(bb) >= 18:
        for idx in [15, 16, 17]:
            if idx < len(bb):
                for p in bb[idx].parameters():
                    p.requires_grad = trainable
        return

    print("[warn] unknown backbone structure for partial unfreeze; unfreezing whole backbone.")
    for p in bb.parameters():
        p.requires_grad = trainable


def _none_if_str(x):
    return None if (x is None or (isinstance(x, str) and x.lower() == "none")) else x



# ORB + BoVW helpers
def _to_u8_img01(img01: np.ndarray) -> np.ndarray:
    img01 = np.asarray(img01, dtype=np.float32)
    return np.clip(img01 * 255.0, 0, 255).astype(np.uint8)


def _to_u8_mask01(mask01: np.ndarray | None) -> np.ndarray | None:
    if mask01 is None:
        return None
    m = np.asarray(mask01, dtype=np.float32)
    return (m > 0.5).astype(np.uint8) * 255


def _extract_orb_descriptors(img01: np.ndarray, mask01: np.ndarray | None, orb: cv2.ORB):
    img_u8 = _to_u8_img01(img01)
    mask_u8 = _to_u8_mask01(mask01)
    _, des = orb.detectAndCompute(img_u8, mask_u8)
    return des


def _bovw_hist(des_u8: np.ndarray | None, kmeans, K: int) -> np.ndarray:
    hist = np.zeros((K,), dtype=np.float32)
    if des_u8 is None or len(des_u8) == 0:
        return hist

    X = des_u8.astype(np.float32)
    idx = kmeans.predict(X)
    hist = np.bincount(idx, minlength=K).astype(np.float32)

    s = float(hist.sum())
    if s > 0:
        hist /= s
    return hist


def train_fold_es_slicelevel(
    fold=0,
    max_epochs=20,
    patience=5,
    lr=1e-4,
    batch_size=32,
    freeze_epochs=3,
    dropout=0.3,
    seed=42,
    grad_clip=1.0,
    wd=2e-4,
    use_aug=True,
    agg_mode="topk",
    topk_frac=0.2,
    topk_k=20,
    mix=0.7,
    backbone="densenet121",
    conv_stem="None",
    attention="None",
    mhsa_tokens=8,
    mhsa_token_dim=64,
    mhsa_heads=4,
    use_high_oversample=False,
    high_raw_thr=4.0,
    high_mult=3.0,
    use_weighted_loss=False,
    high_score_weight=3.0,
    in_ch: int = 3,
    use_orb_bovw: bool = True,
    bovw_k: int = 64,
    orb_nfeatures: int = 500,
    bovw_max_descriptors: int = 50000,
    keypoint_source: str = "medi",
):
    import pandas as pd
    from torch.utils.data import WeightedRandomSampler
    from sklearn.cluster import MiniBatchKMeans

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    index_csv = f"{OUT_ROOT}/train_index.csv"

    train_ds = SliceDatasetRepresentative(
        index_csv=index_csv, fold=fold, split="train",
        seed=seed, center_fraction=CENTER_FRACTION,
        expected_channels=in_ch,
        return_kp=bool(use_orb_bovw),
        return_kp_mask=True,
        keypoint_source=keypoint_source,
    )
    val_ds = SliceDatasetRepresentative(
        index_csv=index_csv, fold=fold, split="val",
        seed=seed, center_fraction=CENTER_FRACTION,
        expected_channels=in_ch,
        return_kp=bool(use_orb_bovw),
        return_kp_mask=True,
        keypoint_source=keypoint_source,
    )

    # oversampling by patient raw label
    if use_high_oversample:
        df = pd.read_csv(index_csv)
        df["patient_id"] = df["patient_id"].astype(str).str.zfill(4)
        df_tr = df[df["fold"] != fold].copy()
        pid_to_raw = dict(zip(df_tr["patient_id"], df_tr["label_raw"]))

        weights = []
        for (pid, y, vp, z, Z) in train_ds.rows:
            raw = float(pid_to_raw.get(pid, 0.0))
            w = float(high_mult) if raw > float(high_raw_thr) else 1.0
            weights.append(w)

        sampler = WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)
        train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, shuffle=False, num_workers=0, drop_last=True)
    else:
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0, drop_last=True)

    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=0)



    # Fit BoVW vocabulary on TRAIN fold only (no leakage)
    # Cache key: (pid, z)
    bovw_cache: dict[tuple[str, int], np.ndarray] = {}

    if use_orb_bovw:
        K = int(bovw_k)
        orb = cv2.ORB_create(nfeatures=int(orb_nfeatures))

        desc_pool = []
        total = 0

        for i in range(len(train_ds)):
            item = train_ds[i]  # x,y,pid,z,kp_img,kp_mask
            kp_img = item[4].numpy()
            kp_mask = item[5].numpy()

            des = _extract_orb_descriptors(kp_img, kp_mask, orb)
            if des is None:
                continue
            desc_pool.append(des)
            total += des.shape[0]
            if total >= int(bovw_max_descriptors):
                break

        if len(desc_pool) == 0:
            print("[warn] ORB found no descriptors in training fold; disabling handcrafted features.")
            use_orb_bovw = False
        else:
            X = np.vstack(desc_pool).astype(np.float32)
            if X.shape[0] > int(bovw_max_descriptors):
                X = X[:int(bovw_max_descriptors)]

            kmeans = MiniBatchKMeans(
                n_clusters=K,
                random_state=int(seed + 1000 + fold),
                batch_size=4096,
                n_init="auto",
                max_iter=200,
            )
            kmeans.fit(X)
            print(f"[BoVW] fitted ORB vocabulary: K={K}, descriptors_used={X.shape[0]}")

            def _fill_cache(ds):
                for i in range(len(ds)):
                    item = ds[i]  # x,y,pid,z,kp_img,kp_mask
                    pid_i = str(item[2])
                    z_i = int(item[3])
                    key = (pid_i, z_i)
                    if key in bovw_cache:
                        continue
                    kp_img = item[4].numpy()
                    kp_mask = item[5].numpy()
                    des = _extract_orb_descriptors(kp_img, kp_mask, orb)
                    bovw_cache[key] = _bovw_hist(des, kmeans, K)

            _fill_cache(train_ds)
            _fill_cache(val_ds)
            print(f"[BoVW] cached histograms: {len(bovw_cache)} slices")

    # backbone + RadImageNet weights
    RAD_PATHS = {
        "resnet50": "/path/to/authorised/data/radimagenet_weights/RadImageNet_pytorch/ResNet50.pt",
        "densenet121": "/path/to/authorised/data/radimagenet_weights/RadImageNet_pytorch/DenseNet121.pt",
        "inceptionv3": "/path/to/authorised/data/radimagenet_weights/RadImageNet_pytorch/InceptionV3.pt",
        "resnet101": None,
        "resnet18": None,
    }
    rad_path = RAD_PATHS.get(backbone, None)

    handcrafted_dim = int(bovw_k) if use_orb_bovw else None

    model = SliceRegressor(
        backbone=backbone,
        in_ch=in_ch,
        imagenet_pretrained=(rad_path is None),
        rad_weight_path=rad_path,
        conv_stem=_none_if_str(conv_stem),
        attention=_none_if_str(attention),
        mhsa_tokens=mhsa_tokens,
        mhsa_token_dim=mhsa_token_dim,
        mhsa_heads=mhsa_heads,

        handcrafted_dim=handcrafted_dim,
        handcrafted_proj=64,
        fusion="concat",
    ).to(device)

    # set dropout
    for m in model.modules():
        if isinstance(m, torch.nn.Dropout):
            m.p = dropout

    # freeze backbone initially
    set_backbone_trainable(model, False)
    for p in model.head.parameters():
        p.requires_grad = True
    for p in model.in_proj.parameters():
        p.requires_grad = True
    if getattr(model, "hand_proj", None) is not None:
        for p in model.hand_proj.parameters():
            p.requires_grad = True

    loss_fn = torch.nn.MSELoss(reduction="none")
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=wd)

    best = {"epoch": -1, "RMSE_log": np.inf, "state": None, "metrics": None}
    bad_epochs = 0
    LOG_THR_FOR_RAW_GT_4 = 0.60206

    for ep in range(1, max_epochs + 1):

        if ep == freeze_epochs + 1:
            set_partial_trainable(model, True)
            opt = torch.optim.AdamW(
                filter(lambda p: p.requires_grad, model.parameters()),
                lr=lr * 0.3,
                weight_decay=wd
            )

        model.train()
        losses = []

        for batch in train_loader:
            x, y, pid, z = batch[0], batch[1], batch[2], batch[3]
            x = x.to(device)
            y = y.to(device)

            if use_aug:
                x = augment_ct_batch(x)

            if use_orb_bovw:
                K = int(bovw_k)
                feats = []
                for i_b in range(len(z)):
                    key = (str(pid[i_b]), int(z[i_b]))
                    f = bovw_cache.get(key, None)
                    if f is None:
                        f = np.zeros((K,), dtype=np.float32)
                    feats.append(f)

                bovw = torch.from_numpy(np.stack(feats, axis=0)).float().to(device)
                pred = model(x, handcrafted=bovw)
            else:
                pred = model(x)

            per_sample_loss = loss_fn(pred, y)

            if use_weighted_loss:
                high_mask = (y > LOG_THR_FOR_RAW_GT_4).float()
                weights = 1.0 + high_mask * (float(high_score_weight) - 1.0)
                loss = (per_sample_loss * weights).mean()
            else:
                loss = per_sample_loss.mean()

            opt.zero_grad()
            loss.backward()
            if grad_clip is not None and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()

            losses.append(float(loss.item()))


        val_metrics = eval_patient_level_from_slices(
            model, val_loader, device,
            agg_mode=agg_mode, topk_frac=topk_frac, topk_k=topk_k, mix=mix,
            compute_raw_metrics=True,
            use_handcrafted=use_orb_bovw,
            bovw_cache=bovw_cache,
            bovw_k=bovw_k,
        )

        print(
            f"Fold {fold} | Epoch {ep:02d} | train MSE {np.mean(losses):.4f} | "
            f"VAL(log) MAE {val_metrics['MAE_log']:.3f} | RMSE {val_metrics['RMSE_log']:.3f} | "
            f"R2 {val_metrics['R2_log']:.3f} | Spearman {val_metrics['Spearman_log']:.3f} || "
            f"VAL(raw) MAE {val_metrics['MAE_raw']:.3f} | RMSE {val_metrics['RMSE_raw']:.3f}"
        )

        if val_metrics["RMSE_log"] + 1e-6 < best["RMSE_log"]:
            best = {
                "epoch": ep,
                "RMSE_log": val_metrics["RMSE_log"],
                "state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "metrics": val_metrics,
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            print(
                f"Early stop: no RMSE_log improvement for {patience} epochs. "
                f"Best epoch={best['epoch']} RMSE_log={best['RMSE_log']:.3f}"
            )
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"], strict=True)

    return best["metrics"], best["epoch"], model




In [ ]:
# Cell 13 — 5-fold CV (slice-level training, patient-level evaluation) + save best model per fold

import os
from pathlib import Path
import pandas as pd
import numpy as np
import torch

results = []
best_epochs = []

# ----------------------------
# Save config
# ----------------------------
SAVE_DIR = Path("checkpoints/resnet18_3ch_dog_orbbovw_k64")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ----------------------------
# Model config
# ----------------------------
BACKBONE   = "resnet18"
CONV_STEM  = None      # None | "dwsep" | "1x1_5x5"
ATTENTION  = None      # None | "se" | "mhsa"

# Input channels (from preprocess_slice)
IN_CH = 3

# Only used if ATTENTION == "mhsa"
MHSA_TOKENS     = 8
MHSA_TOKEN_DIM  = 64
MHSA_HEADS      = 4

# ----------------------------
# ORB-BoVW fusion config
# ----------------------------
USE_ORB_BOVW      = True
BOVW_K            = 64
ORB_NFEATURES     = 500
BOVW_MAX_DESC     = 50000
KEYPOINT_SOURCE   = "medi"

# ----------------------------
# Training config
# ----------------------------
MAX_EPOCHS    = 30
PATIENCE      = 6
LR            = 1e-4
BATCH_SIZE    = 32
FREEZE_EPOCHS = 2
DROPOUT       = 0.3
SEED          = 42
WD            = 2e-4

# ----------------------------
# Patient aggregation for validation / early stopping
# ----------------------------
AGG_MODE  = "mix"
MIX       = 0.7
TOPK_K    = 20
TOPK_FRAC = 0.15

for fold in range(5):
    metrics, best_ep, model = train_fold_es_slicelevel(
        fold=fold,

        # model knobs
        backbone=BACKBONE,
        conv_stem=CONV_STEM,
        attention=ATTENTION,
        in_ch=IN_CH,

        # mhsa knobs
        mhsa_tokens=MHSA_TOKENS,
        mhsa_token_dim=MHSA_TOKEN_DIM,
        mhsa_heads=MHSA_HEADS,

        # training knobs
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        lr=LR,
        batch_size=BATCH_SIZE,
        freeze_epochs=FREEZE_EPOCHS,
        dropout=DROPOUT,
        seed=SEED,
        wd=WD,
        use_aug=True,

        # patient aggregation for validation / early stopping
        agg_mode=AGG_MODE,
        mix=MIX,
        topk_k=TOPK_K,
        topk_frac=TOPK_FRAC,

        # clean comparisons
        use_weighted_loss=False,
        high_score_weight=3.0,
        use_high_oversample=False,
        high_raw_thr=4.0,
        high_mult=3.0,

        # ORB-BoVW knobs
        use_orb_bovw=USE_ORB_BOVW,
        bovw_k=BOVW_K,
        orb_nfeatures=ORB_NFEATURES,
        bovw_max_descriptors=BOVW_MAX_DESC,
        keypoint_source=KEYPOINT_SOURCE,
    )

    # log preprocessing HF settings for reproducibility
    hf_cfg = dict(
        add_hf_channel=bool(ADD_HF_CHANNEL),
        hf_mode=str(HF_MODE),
        hf_from=str(HF_FROM),
        hf_gauss_sigma=float(HF_GAUSS_SIGMA),
        dog_sigma1=float(globals().get("DOG_SIGMA1", np.nan)),
        dog_sigma2=float(globals().get("DOG_SIGMA2", np.nan)),
        hf_log_sigma=float(HF_LOG_SIGMA),
        dct_low_cutoff=float(globals().get("DCT_LOW_CUTOFF", np.nan)),
        dct_high_cutoff=float(globals().get("DCT_HIGH_CUTOFF", np.nan) if globals().get("DCT_HIGH_CUTOFF", None) is not None else np.nan),
        dct_use_abs=bool(globals().get("DCT_USE_ABS", False)),
        hf_pctl_low=float(HF_PCTL_LOW),
        hf_pctl_high=float(HF_PCTL_HIGH),
        use_lung_mask_for_hf=bool(globals().get("USE_LUNG_MASK_FOR_HF", False)),
    )

    # log ORB-BoVW settings for reproducibility
    orb_cfg = dict(
        use_orb_bovw=bool(USE_ORB_BOVW),
        bovw_k=int(BOVW_K),
        orb_nfeatures=int(ORB_NFEATURES),
        bovw_max_descriptors=int(BOVW_MAX_DESC),
        keypoint_source=str(KEYPOINT_SOURCE),
    )

    row = {
        "fold": fold,
        "best_epoch": best_ep,
        "backbone": BACKBONE,
        "in_ch": IN_CH,
        "conv_stem": str(CONV_STEM),
        "attention": str(ATTENTION),
        "mhsa_tokens": (MHSA_TOKENS if ATTENTION == "mhsa" else None),
        "mhsa_token_dim": (MHSA_TOKEN_DIM if ATTENTION == "mhsa" else None),
        "mhsa_heads": (MHSA_HEADS if ATTENTION == "mhsa" else None),
        **hf_cfg,
        **orb_cfg,
        **metrics,
    }
    results.append(row)
    best_epochs.append(best_ep)

    # ----------------------------
    # Save best model checkpoint for this fold
    # ----------------------------
    ckpt_path = SAVE_DIR / f"fold{fold}_best.pt"
    torch.save({
        "fold": fold,
        "best_epoch": best_ep,
        "state_dict": model.state_dict(),
        "metrics": metrics,
        "model_config": {
            "backbone": BACKBONE,
            "conv_stem": CONV_STEM,
            "attention": ATTENTION,
            "in_ch": IN_CH,
            "mhsa_tokens": MHSA_TOKENS,
            "mhsa_token_dim": MHSA_TOKEN_DIM,
            "mhsa_heads": MHSA_HEADS,
            "dropout": DROPOUT,
        },
        "train_config": {
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "lr": LR,
            "batch_size": BATCH_SIZE,
            "freeze_epochs": FREEZE_EPOCHS,
            "seed": SEED,
            "weight_decay": WD,
            "use_aug": True,
            "agg_mode": AGG_MODE,
            "mix": MIX,
            "topk_k": TOPK_K,
            "topk_frac": TOPK_FRAC,
        },
        "hf_config": hf_cfg,
        "orb_bovw_config": orb_cfg,
    }, ckpt_path)

    print(f"\nFold {fold} best epoch: {best_ep}")
    print("  LOG:",
          {k: metrics[k] for k in ["MAE_log","RMSE_log","R2_log","Spearman_log","Pearson_log"] if k in metrics})
    print("  RAW:",
          {k: metrics[k] for k in ["MAE_raw","RMSE_raw","R2_raw","Spearman_raw","Pearson_raw"] if k in metrics})
    print(f"  Saved checkpoint to: {ckpt_path}")

df_res = pd.DataFrame(results)

# Save fold metrics table too
csv_path = SAVE_DIR / "cv_results.csv"
df_res.to_csv(csv_path, index=False)

print("\nPer-fold results:")
display(df_res)

def summarize(cols, title):
    print(f"\nSummary ({title}) mean ± std:")
    for col in cols:
        if col not in df_res.columns:
            continue
        m = float(np.nanmean(df_res[col].values))
        s = float(np.nanstd(df_res[col].values, ddof=1))
        print(f"{col}: {m:.3f} ± {s:.3f}")

summarize(["MAE_log","RMSE_log","R2_log","Spearman_log","Pearson_log"], "LOG")
summarize(["MAE_raw","RMSE_raw","R2_raw","Spearman_raw","Pearson_raw"], "RAW")
print("\nBest epochs per fold:", best_epochs)
print(f"Saved CV results to: {csv_path}")



In [ ]:
# Cell C — Fine-tuning on MosMed (Fusion: ORB-BoVW + CNN)
# Source: internal pretrained checkpoint -> Target: MosMed train/val/test
# Staged schedule: head-only -> unfreeze layer4 -> unfreeze layer3
# IMPORTANT: BoVW vocabulary is fit on MosMed TRAIN only (no leakage).

import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import cv2
from scipy.stats import spearmanr
from sklearn.cluster import MiniBatchKMeans


# -----------------------
# Paths / init
# -----------------------
PROJECT_ROOT = DATA_ROOT
MOSMED_SPLIT = os.path.join(PROJECT_ROOT, "external_data/mosmed/mosmed_split.csv")
SOURCE_CKPT  = os.path.join(PROJECT_ROOT, "checkpoints/resnet18_3ch_orbbovw_k64/fold2_best.pt")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# -----------------------
# BoVW knobs (MosMed-only)
# -----------------------
USE_ORB_BOVW        = True
BOVW_K              = 64          # 64 first; later test 128
ORB_NFEATURES       = 500         # max keypoints per slice
BOVW_MAX_DESCRIPTORS= 50000       # cap for fitting kmeans
KEYPOINT_SOURCE     = "medi"      # "medi" recommended, can test "lung"
USE_KP_MASK         = True        # use lung mask to restrict keypoints


# -----------------------
# Data
# -----------------------
BATCH_SIZE      = 32
BATCH_SIZE_EVAL = 64

# return_kp=True is required for fitting BoVW (train only).
# We keep val/test returning kp too because it simplifies caching.
ds_tr = MosMedSliceDatasetSplit(
    MOSMED_SPLIT, split="train", expected_channels=3,
    return_kp=USE_ORB_BOVW, return_kp_mask=USE_KP_MASK, keypoint_source=KEYPOINT_SOURCE
)
ds_va = MosMedSliceDatasetSplit(
    MOSMED_SPLIT, split="val", expected_channels=3,
    return_kp=USE_ORB_BOVW, return_kp_mask=USE_KP_MASK, keypoint_source=KEYPOINT_SOURCE
)
ds_te = MosMedSliceDatasetSplit(
    MOSMED_SPLIT, split="test", expected_channels=3,
    return_kp=USE_ORB_BOVW, return_kp_mask=USE_KP_MASK, keypoint_source=KEYPOINT_SOURCE
)

# DataLoaders
# If return_kp=True: batch = (x,y,pid,z,kp_img,kp_mask)
# Else:             batch = (x,y,pid,z)
tr_loader = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
va_loader = DataLoader(ds_va, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=0)
te_loader = DataLoader(ds_te, batch_size=BATCH_SIZE_EVAL, shuffle=False, num_workers=0)

print("MosMed slices | train:", len(ds_tr), "| val:", len(ds_va), "| test:", len(ds_te))


# -----------------------
# ORB-BoVW helpers
# -----------------------
def _to_u8_img01(img01: np.ndarray) -> np.ndarray:
    img01 = np.asarray(img01, dtype=np.float32)
    return np.clip(img01 * 255.0, 0, 255).astype(np.uint8)

def _to_u8_mask01(mask01: np.ndarray | None) -> np.ndarray | None:
    if mask01 is None:
        return None
    m = np.asarray(mask01, dtype=np.float32)
    return (m > 0.5).astype(np.uint8) * 255

def _extract_orb_descriptors(img01: np.ndarray, mask01: np.ndarray | None, orb: cv2.ORB):
    img_u8 = _to_u8_img01(img01)
    mask_u8 = _to_u8_mask01(mask01)
    _, des = orb.detectAndCompute(img_u8, mask_u8)
    return des  # None if no keypoints

def _bovw_hist(des_u8: np.ndarray | None, kmeans, K: int) -> np.ndarray:
    hist = np.zeros((K,), dtype=np.float32)
    if des_u8 is None or len(des_u8) == 0:
        return hist
    X = des_u8.astype(np.float32)
    idx = kmeans.predict(X)
    hist = np.bincount(idx, minlength=K).astype(np.float32)
    s = float(hist.sum())
    if s > 0:
        hist /= s
    return hist


# -----------------------
# Fit BoVW vocabulary on TRAIN only + cache histograms (pid,z) -> K
# -----------------------
bovw_cache: dict[tuple[str, int], np.ndarray] = {}
kmeans = None

if USE_ORB_BOVW:
    K = int(BOVW_K)
    orb = cv2.ORB_create(nfeatures=int(ORB_NFEATURES))

    # collect descriptors from TRAIN fold only
    desc_pool = []
    total = 0

    for i in range(len(ds_tr)):
        item = ds_tr[i]
        # item: x,y,pid,z,kp_img,kp_mask
        kp_img = item[4].numpy()
        kp_mask = item[5].numpy() if (USE_KP_MASK and len(item) >= 6) else None

        des = _extract_orb_descriptors(kp_img, kp_mask, orb)
        if des is None:
            continue
        desc_pool.append(des)
        total += des.shape[0]
        if total >= int(BOVW_MAX_DESCRIPTORS):
            break

    if len(desc_pool) == 0:
        print("[warn] ORB found no descriptors on MosMed train; disabling fusion.")
        USE_ORB_BOVW = False
    else:
        X = np.vstack(desc_pool).astype(np.float32)
        if X.shape[0] > int(BOVW_MAX_DESCRIPTORS):
            X = X[:int(BOVW_MAX_DESCRIPTORS)]

        kmeans = MiniBatchKMeans(
            n_clusters=K,
            random_state=1337,
            batch_size=4096,
            n_init="auto",
            max_iter=200,
        )
        kmeans.fit(X)
        print(f"[BoVW MosMed] fitted vocabulary: K={K}, descriptors_used={X.shape[0]}")

        # cache histograms for all splits (train/val/test)
        def _fill_cache(ds):
            for i in range(len(ds)):
                item = ds[i]
                pid_i = str(item[2])
                z_i = int(item[3])
                key = (pid_i, z_i)
                if key in bovw_cache:
                    continue
                kp_img = item[4].numpy()
                kp_mask = item[5].numpy() if (USE_KP_MASK and len(item) >= 6) else None
                des = _extract_orb_descriptors(kp_img, kp_mask, orb)
                bovw_cache[key] = _bovw_hist(des, kmeans, K)

        _fill_cache(ds_tr)
        _fill_cache(ds_va)
        _fill_cache(ds_te)
        print(f"[BoVW MosMed] cached histograms: {len(bovw_cache)} slices")


# -----------------------
# Metrics (MosMed labels numeric 0..4)
# -----------------------
def regression_metrics_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

    sse = float(np.sum((y_true - y_pred) ** 2))
    sst = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - sse / (sst + 1e-12))

    rho = spearmanr(y_true, y_pred).correlation
    rho = 0.0 if (rho is None or np.isnan(rho)) else float(rho)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "Spearman": rho}


def eval_mosmed_patient_level(model, loader, device, agg_mode="mix", mix=0.7, topk_k=20, topk_frac=0.15, use_bovw=False):
    model.eval()
    preds_by_pid = {}
    y_by_pid = {}

    with torch.no_grad():
        for batch in loader:
            # with kp outputs: x,y,pid,z,kp_img,kp_mask
            x, y, pid, z = batch[0], batch[1], batch[2], batch[3]
            x = x.to(device)

            if use_bovw:
                K = int(BOVW_K)
                feats = []
                for i_b in range(len(z)):
                    key = (str(pid[i_b]), int(z[i_b]))
                    f = bovw_cache.get(key, None)
                    if f is None:
                        f = np.zeros((K,), dtype=np.float32)
                    feats.append(f)
                bovw = torch.from_numpy(np.stack(feats, axis=0)).float().to(device)
                preds = model(x, handcrafted=bovw).detach().cpu().numpy().astype(float)
            else:
                preds = model(x).detach().cpu().numpy().astype(float)

            y_np = y.detach().cpu().numpy().astype(float)

            for i in range(len(preds)):
                p = str(pid[i])
                preds_by_pid.setdefault(p, []).append(float(preds[i]))
                y_by_pid[p] = float(y_np[i])

    def aggregate_scores(scores):
        scores = np.asarray(scores, dtype=float)
        if len(scores) == 0:
            return 0.0
        if agg_mode == "mean":
            return float(scores.mean())
        if topk_k is not None:
            k = int(topk_k)
        else:
            k = int(max(1, int(len(scores) * float(topk_frac))))
        k = max(1, min(k, len(scores)))
        top = float(np.sort(scores)[-k:].mean())
        if agg_mode == "topk":
            return top
        if agg_mode == "mix":
            return float(mix * top + (1.0 - mix) * scores.mean())
        raise ValueError("agg_mode must be mean/topk/mix")

    pids = sorted(preds_by_pid.keys())
    y_true = np.array([y_by_pid[p] for p in pids], dtype=float)
    y_pred = np.array([aggregate_scores(preds_by_pid[p]) for p in pids], dtype=float)
    return regression_metrics_np(y_true, y_pred)


# -----------------------
# Model init from source checkpoint
# -----------------------
handcrafted_dim = int(BOVW_K) if USE_ORB_BOVW else None

model = SliceRegressor(
    backbone="resnet18",
    in_ch=3,
    imagenet_pretrained=False,
    handcrafted_dim=handcrafted_dim,
    handcrafted_proj=64,
    fusion="concat",
).to(device)

ckpt = torch.load(SOURCE_CKPT, map_location="cpu")
model.load_state_dict(ckpt["state_dict"], strict=True)
print("[init] loaded source weights from:", SOURCE_CKPT)


# -----------------------
# Fine-tune stages
# -----------------------
LOSS = torch.nn.MSELoss()

def set_requires_grad(module, req: bool):
    for p in module.parameters():
        p.requires_grad = req

def unfreeze_resnet_layer(model, layer_name: str, req: bool = True):
    bb = model.backbone
    if not hasattr(bb, layer_name):
        raise ValueError(f"Backbone has no {layer_name}")
    set_requires_grad(getattr(bb, layer_name), req)

def train_stage(stage_name, epochs, lr, wd=2e-4, grad_clip=1.0, patience=2):
    """
    Train for up to `epochs`, but early-stop if val RMSE doesn't improve for `patience` epochs.
    Restores the best state within this stage.
    """
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=wd)
    best = {"rmse": np.inf, "state": None, "epoch": -1, "metrics": None}
    bad = 0

    for ep in range(1, epochs + 1):
        model.train()
        losses = []

        for batch in tr_loader:
            x, y, pid, z = batch[0], batch[1], batch[2], batch[3]
            x = x.to(device)
            y = y.to(device)

            if USE_ORB_BOVW:
                K = int(BOVW_K)
                feats = []
                for i_b in range(len(z)):
                    key = (str(pid[i_b]), int(z[i_b]))
                    f = bovw_cache.get(key, None)
                    if f is None:
                        f = np.zeros((K,), dtype=np.float32)
                    feats.append(f)
                bovw = torch.from_numpy(np.stack(feats, axis=0)).float().to(device)
                pred = model(x, handcrafted=bovw)
            else:
                pred = model(x)

            loss = LOSS(pred, y)
            opt.zero_grad()
            loss.backward()
            if grad_clip and grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            opt.step()
            losses.append(float(loss.item()))

        val_m = eval_mosmed_patient_level(
            model, va_loader, device,
            agg_mode="mix", mix=0.7, topk_k=20, topk_frac=0.15,
            use_bovw=USE_ORB_BOVW
        )

        print(f"[{stage_name}] Ep {ep:02d}/{epochs} | train MSE {np.mean(losses):.4f} | "
              f"VAL MAE {val_m['MAE']:.3f} RMSE {val_m['RMSE']:.3f} R2 {val_m['R2']:.3f} Spearman {val_m['Spearman']:.3f}")

        # early stopping logic (val RMSE)
        if val_m["RMSE"] + 1e-6 < best["rmse"]:
            best["rmse"] = val_m["RMSE"]
            best["epoch"] = ep
            best["metrics"] = val_m
            best["state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1

        if bad >= int(patience):
            print(f"[{stage_name}] early stop: no val RMSE improvement for {patience} epochs "
                  f"(best ep={best['epoch']}, best RMSE={best['rmse']:.3f})")
            break

    if best["state"] is not None:
        model.load_state_dict(best["state"], strict=True)

    print(f"[{stage_name}] best epoch={best['epoch']} val_RMSE={best['rmse']:.3f}")
    return best


# Stage 1: head-only (also trains hand_proj if fusion is enabled)
set_requires_grad(model.backbone, False)
set_requires_grad(model.head, True)
set_requires_grad(model.in_proj, True)
if getattr(model, "hand_proj", None) is not None:
    set_requires_grad(model.hand_proj, True)

best_s1 = train_stage("head_only", epochs=3, lr=1e-3)

# Stage 2: unfreeze layer4
unfreeze_resnet_layer(model, "layer4", True)
best_s2 = train_stage("unfreeze_layer4", epochs=8, lr=3e-5, patience=3)

# Stage 3: unfreeze layer3 (optional)
DO_STAGE3 = True
best_s3 = None
if DO_STAGE3:
    unfreeze_resnet_layer(model, "layer3", True)
    best_s3 = train_stage("unfreeze_layer3", epochs=6, lr=1e-5, patience=3)


# Final evaluation on MosMed TEST ONLY
test_m = eval_mosmed_patient_level(
    model, te_loader, device,
    agg_mode="mix", mix=0.7, topk_k=20, topk_frac=0.15,
    use_bovw=USE_ORB_BOVW
)
print("\n[MOSMED TEST] Fusion (ORB-BoVW + CNN) fine-tuned from source checkpoint")
print(test_m)


# Save fine-tuned weights + BoVW metadata
OUT_DIR = os.path.join(PROJECT_ROOT, "checkpoints", "mosmed_finetune_fusion")
os.makedirs(OUT_DIR, exist_ok=True)
OUT_PATH = os.path.join(OUT_DIR, "fusion_orb_bovw_finetuned_from_source_fold1.pt")

save_obj = {
    "source_ckpt": SOURCE_CKPT,
    "mosmed_split": MOSMED_SPLIT,
    "use_orb_bovw": bool(USE_ORB_BOVW),
    "bovw_k": int(BOVW_K) if USE_ORB_BOVW else None,
    "orb_nfeatures": int(ORB_NFEATURES) if USE_ORB_BOVW else None,
    "bovw_max_descriptors": int(BOVW_MAX_DESCRIPTORS) if USE_ORB_BOVW else None,
    "keypoint_source": str(KEYPOINT_SOURCE),
    "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
    "test_metrics": test_m,
    "stage1_best": best_s1,
    "stage2_best": best_s2,
    "stage3_best": best_s3,
}

# Storing full kmeans object is optional; it can be large. Usually you only need config + results.
torch.save(save_obj, OUT_PATH)
print("[saved]", OUT_PATH)



In [ ]:
# Cell Z — Fusion WITHOUT fine-tuning (zero-shot) on MosMed TEST
# Uses: Source fusion checkpoint + Source BoVW vocabulary (fit on source-train fold only)
# Evaluates on MosMed TEST only. No training.

import os, numpy as np, torch, cv2
from torch.utils.data import DataLoader
from sklearn.cluster import MiniBatchKMeans
from scipy.stats import spearmanr

PROJECT_ROOT = DATA_ROOT
INDEX_CSV    = os.path.join(PROJECT_ROOT, "train_index.csv")
MOSMED_SPLIT = os.path.join(PROJECT_ROOT, "external_data/mosmed/mosmed_split.csv")

# Choose which source fold checkpoint to use
SOURCE_FOLD = 2
SOURCE_CKPT = os.path.join(PROJECT_ROOT, f"checkpoints/resnet18_3ch_orbbovw_k64/fold{SOURCE_FOLD}_best.pt")

# BoVW settings (MUST match what you used in source training)
BOVW_K = 64
ORB_NFEATURES = 500
BOVW_MAX_DESCRIPTORS = 50000
KEYPOINT_SOURCE = "medi"
USE_KP_MASK = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- helpers ----------
def regression_metrics_np(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    sse = float(np.sum((y_true - y_pred) ** 2))
    sst = float(np.sum((y_true - np.mean(y_true)) ** 2))
    r2 = float(1.0 - sse / (sst + 1e-12))
    rho = spearmanr(y_true, y_pred).correlation
    rho = 0.0 if (rho is None or np.isnan(rho)) else float(rho)
    return {"MAE": mae, "RMSE": rmse, "R2": r2, "Spearman": rho}

def _to_u8_img01(img01):
    img01 = np.asarray(img01, dtype=np.float32)
    return np.clip(img01 * 255.0, 0, 255).astype(np.uint8)

def _to_u8_mask01(mask01):
    if mask01 is None:
        return None
    m = np.asarray(mask01, dtype=np.float32)
    return (m > 0.5).astype(np.uint8) * 255

def _extract_orb_descriptors(img01, mask01, orb):
    img_u8 = _to_u8_img01(img01)
    mask_u8 = _to_u8_mask01(mask01)
    _, des = orb.detectAndCompute(img_u8, mask_u8)
    return des

def _bovw_hist(des_u8, kmeans, K):
    hist = np.zeros((K,), dtype=np.float32)
    if des_u8 is None or len(des_u8) == 0:
        return hist
    X = des_u8.astype(np.float32)
    idx = kmeans.predict(X)
    hist = np.bincount(idx, minlength=K).astype(np.float32)
    s = float(hist.sum())
    if s > 0:
        hist /= s
    return hist

def eval_mosmed_patient_level_fusion(model, loader, bovw_cache, K, device,
                                     agg_mode="mix", mix=0.7, topk_k=20, topk_frac=0.15):
    model.eval()
    preds_by_pid, y_by_pid = {}, {}
    with torch.no_grad():
        for batch in loader:
            x, y, pid, z = batch[0], batch[1], batch[2], batch[3]
            x = x.to(device)

            feats = []
            for i_b in range(len(z)):
                key = (str(pid[i_b]), int(z[i_b]))
                f = bovw_cache.get(key, None)
                if f is None:
                    f = np.zeros((K,), dtype=np.float32)
                feats.append(f)
            bovw = torch.from_numpy(np.stack(feats, axis=0)).float().to(device)

            preds = model(x, handcrafted=bovw).detach().cpu().numpy().astype(float)
            y_np  = y.detach().cpu().numpy().astype(float)

            for i in range(len(preds)):
                p = str(pid[i])
                preds_by_pid.setdefault(p, []).append(float(preds[i]))
                y_by_pid[p] = float(y_np[i])

    def aggregate_scores(scores):
        scores = np.asarray(scores, dtype=float)
        if len(scores) == 0:
            return 0.0
        if agg_mode == "mean":
            return float(scores.mean())
        if topk_k is not None:
            k = int(topk_k)
        else:
            k = int(max(1, int(len(scores) * float(topk_frac))))
        k = max(1, min(k, len(scores)))
        top = float(np.sort(scores)[-k:].mean())
        if agg_mode == "topk":
            return top
        if agg_mode == "mix":
            return float(mix * top + (1.0 - mix) * scores.mean())
        raise ValueError("agg_mode must be mean/topk/mix")

    pids = sorted(preds_by_pid.keys())
    y_true = np.array([y_by_pid[p] for p in pids], dtype=float)
    y_pred = np.array([aggregate_scores(preds_by_pid[p]) for p in pids], dtype=float)
    return regression_metrics_np(y_true, y_pred)

# ---------- 1) Fit SOURCE kmeans on SOURCE TRAIN only (fold != SOURCE_FOLD) ----------
orb = cv2.ORB_create(nfeatures=int(ORB_NFEATURES))

src_ds = SliceDatasetRepresentative(
    index_csv=INDEX_CSV,
    fold=SOURCE_FOLD,
    split="train",
    seed=42,
    expected_channels=3,
    return_kp=True,
    return_kp_mask=True,
    keypoint_source=KEYPOINT_SOURCE,
)

desc_pool, total = [], 0
for i in range(len(src_ds)):
    item = src_ds[i]  # x,y,pid,z,kp_img,kp_mask
    kp_img = item[4].numpy()
    kp_mask = item[5].numpy() if USE_KP_MASK else None
    des = _extract_orb_descriptors(kp_img, kp_mask, orb)
    if des is None:
        continue
    desc_pool.append(des)
    total += des.shape[0]
    if total >= int(BOVW_MAX_DESCRIPTORS):
        break

X = np.vstack(desc_pool).astype(np.float32)
if X.shape[0] > int(BOVW_MAX_DESCRIPTORS):
    X = X[:int(BOVW_MAX_DESCRIPTORS)]

kmeans = MiniBatchKMeans(
    n_clusters=int(BOVW_K),
    random_state=1337 + SOURCE_FOLD,
    batch_size=4096,
    n_init="auto",
    max_iter=200,
)
kmeans.fit(X)
print(f"[source BoVW] fitted: fold={SOURCE_FOLD}, K={BOVW_K}, descriptors_used={X.shape[0]}")

# ---------- 2) Build MosMed TEST loader WITH kp outputs ----------
ds_te = MosMedSliceDatasetSplit(
    MOSMED_SPLIT, split="test", expected_channels=3,
    return_kp=True, return_kp_mask=True, keypoint_source=KEYPOINT_SOURCE
)
te_loader = DataLoader(ds_te, batch_size=64, shuffle=False, num_workers=0)
print("MosMed test slices:", len(ds_te))

# ---------- 3) Cache MosMed BoVW histograms using SOURCE kmeans ----------
bovw_cache = {}
for i in range(len(ds_te)):
    item = ds_te[i]  # x,y,pid,z,kp_img,kp_mask
    pid_i = str(item[2]); z_i = int(item[3])
    kp_img = item[4].numpy()
    kp_mask = item[5].numpy() if USE_KP_MASK else None
    des = _extract_orb_descriptors(kp_img, kp_mask, orb)
    bovw_cache[(pid_i, z_i)] = _bovw_hist(des, kmeans, int(BOVW_K))
print("[MosMed test] cached BoVW:", len(bovw_cache))

# ---------- 4) Load SOURCE fusion checkpoint and evaluate (NO training) ----------
model = SliceRegressor(
    backbone="resnet18",
    in_ch=3,
    imagenet_pretrained=False,
    handcrafted_dim=int(BOVW_K),
    handcrafted_proj=64,
    fusion="concat",
).to(device)

ckpt = torch.load(SOURCE_CKPT, map_location="cpu")
model.load_state_dict(ckpt["state_dict"], strict=True)
print("[init] loaded source fusion weights:", SOURCE_CKPT)

test_m = eval_mosmed_patient_level_fusion(model, te_loader, bovw_cache, int(BOVW_K), device)
print("\n[MOSMED TEST] Fusion WITHOUT fine-tuning (zero-shot)")
print(test_m)
